# 🚗 Araç Hasar Tespit Sistemi — Tez Pipeline

> Sigorta sektöründe hasar tespitinin derin öğrenme ile otomasyonu

---

## 🏗️ Sistem Mimarisi

```
                    [Hasarlı Araç Fotoğrafı]
                              │
              ┌───────────────┴───────────────┐
              ▼                               ▼
   ┌────────────────────┐         ┌────────────────────┐
   │  MODEL 1: KONUM    │         │  MODEL 2: HASAR    │
   │  Carparts-Seg      │         │  Kendi Verim       │
   │  23 → 8 parça      │         │  cizik/gocuk/kirik │
   └────────────────────┘         └────────────────────┘
              │                               │
              └───────────────┬───────────────┘
                              ▼
                  ┌───────────────────────┐
                  │   FÜZYON (IoU Eşleme) │
                  │  Tampon-Kirik         │
                  │  Camurluk-Cizik       │
                  └───────────────────────┘
                              │
              ┌───────────────┴───────────────┐
              ▼                               ▼
     ┌─────────────────┐            ┌──────────────────┐
     │  GÖRSEL ÇIKTI   │            │  TRAMER RAPORU   │
     │  Maskeli overlay│            │  PDF belge       │
     └─────────────────┘            └──────────────────┘
```

---

## 📊 Tez İçin Model Karşılaştırması (5 Model)

| Model | Amaç |
|---|---|
| YOLOv11n-seg | Hız baseline — edge/mobil senaryosu |
| YOLOv11s-seg | Küçük-orta denge |
| YOLOv11m-seg | Ana aday model |
| YOLOv11l-seg | Doğruluk üst sınırı |
| YOLOv8m-seg | Önceki nesil referansı |

Hepsi **aynı test setinde** karşılaştırılır → mAP50, mAP50-95, inference süresi, model boyutu tablosu.

---

## ⚙️ Çalıştırma Sırası

| Hücre | İçerik | Süre |
|---|---|---|
| 1 | Kurulum + ayarlar | 1 dk |
| 2 | Model 1 (Konum) veri hazırlığı | 2 dk |
| 3 | Model 1 eğitimi | ~40 dk |
| 4 | Model 2 veri indirme + temizlik | 5 dk |
| 5 | **5 MODEL KARŞILAŞTIRMALI EĞİTİM** | ~4-5 saat |
| 6 | Karşılaştırma tablosu + grafikler | 2 dk |
| 7 | Modelleri yükle + füzyon fonksiyonları | 1 dk |
| 8 | Tek görsel testi | 30 sn |
| 9 | Tramer raporu (PDF) | 1 dk |
| 10 | Toplu işlem + Excel özet | Değişken |

---

> ⚡ **Colab Pro:** Runtime → Change runtime type → **A100 GPU** (varsa) veya **V100/T4**

---
## 💾 HÜCRE 0: Google Drive Bağlantısı (ÖNEMLİ — İLK ÇALIŞTIR!)

⚠️ **Colab session kapandığında `/content` klasörü tamamen silinir.**
Bu hücre tüm model ve çıktıları Google Drive'a kaydeder — böylece eğitim bir kez yapılır.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# 💾 HÜCRE 0: GOOGLE DRIVE BAĞLANTISI
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NEDEN GEREKLİ?
#    Colab session kapanınca /content silinir. Eğitilen modeller kaybolur.
#    Bu hücre her şeyi Drive'a kaydeder → bir kez eğit, sonsuza kadar kullan.
#
# 📁 DRIVE KLASÖR YAPISI:
#    MyDrive/tez_arac_hasar/
#       ├── modeller/          ← Eğitilen tüm modeller (.pt)
#       ├── ciktilar/          ← Grafikler, Excel, PDF raporlar
#       └── karsilastirma_sonuclari.json
#
# 💡 KULLANIM:
#    - İlk çalıştırmada: Drive erişim izni ver
#    - Sonraki session'larda: Aynı hücreyi çalıştır, modeller otomatik bulunur
# ═══════════════════════════════════════════════════════════════════════════════

import os

DRIVE_KULLAN = True   # False yaparsan sadece geçici diske kaydeder

if DRIVE_KULLAN:
    try:
        from google.colab import drive
        drive.mount('/content/drive')

        ANA_DIZIN = '/content/drive/MyDrive/tez_arac_hasar'
        print(f'✅ Drive bağlandı')
    except Exception as e:
        print(f'⚠️ Drive bağlanamadı: {str(e)[:80]}')
        print('   Geçici diske kaydedilecek (session kapanınca silinir!)')
        ANA_DIZIN = '/content/tez_arac_hasar'
else:
    ANA_DIZIN = '/content/tez_arac_hasar'
    print('⚠️ Drive kullanılmıyor — modeller session sonunda silinecek!')

# ─── KLASÖRLERİ OLUŞTUR ───────────────────────────────────────────────────────
MODEL_DIZIN = os.path.join(ANA_DIZIN, 'modeller')
OUTPUT_DIR  = os.path.join(ANA_DIZIN, 'ciktilar')
RAPOR_DIR   = os.path.join(OUTPUT_DIR, 'raporlar')
GORSEL_DIR  = os.path.join(OUTPUT_DIR, 'gorseller')

for d in [ANA_DIZIN, MODEL_DIZIN, OUTPUT_DIR, RAPOR_DIR, GORSEL_DIR]:
    os.makedirs(d, exist_ok=True)

SONUC_JSON = os.path.join(ANA_DIZIN, 'karsilastirma_sonuclari.json')

print()
print('=' * 70)
print('📁 KLASÖR YAPISI')
print('=' * 70)
print(f'   Ana dizin : {ANA_DIZIN}')
print(f'   Modeller  : {MODEL_DIZIN}')
print(f'   Çıktılar  : {OUTPUT_DIR}')
print(f'   Sonuç JSON: {SONUC_JSON}')

# ─── MEVCUT MODELLERİ KONTROL ET ──────────────────────────────────────────────
print()
print('=' * 70)
print('🔍 MEVCUT MODELLER')
print('=' * 70)

import glob, json as _json

bulunan_modeller = glob.glob(f'{MODEL_DIZIN}/**/weights/best.pt', recursive=True)
if bulunan_modeller:
    for mp in sorted(bulunan_modeller):
        ad = mp.split('/')[-3]
        boyut = os.path.getsize(mp) / 1e6
        print(f'   ✅ {ad:30s} ({boyut:.1f} MB)')
    print(f'\n   💡 Bu modeller tekrar eğitilmeyecek — Hücre 5 otomatik atlar.')
else:
    print('   (henüz model yok — Hücre 3 ve 5 ile eğitilecek)')

if os.path.exists(SONUC_JSON):
    with open(SONUC_JSON) as f:
        _s = _json.load(f)
    print(f'\n   📊 Kayıtlı karşılaştırma sonucu: {len(_s)} model')
    for k, v in _s.items():
        print(f'      {v["ad"]:20s} mAP50={v["mask_mAP50"]:.4f}')

print('=' * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive bağlandı

📁 KLASÖR YAPISI
   Ana dizin : /content/drive/MyDrive/tez_arac_hasar
   Modeller  : /content/drive/MyDrive/tez_arac_hasar/modeller
   Çıktılar  : /content/drive/MyDrive/tez_arac_hasar/ciktilar
   Sonuç JSON: /content/drive/MyDrive/tez_arac_hasar/karsilastirma_sonuclari.json

🔍 MEVCUT MODELLER
   ✅ hasar_yolo11l-seg              (55.8 MB)
   ✅ hasar_yolo11m-seg              (45.2 MB)
   ✅ hasar_yolo11n-seg              (6.0 MB)
   ✅ hasar_yolo11s-seg              (20.5 MB)
   ✅ hasar_yolov8m-seg              (54.8 MB)
   ✅ konum                          (45.2 MB)

   💡 Bu modeller tekrar eğitilmeyecek — Hücre 5 otomatik atlar.

   📊 Kayıtlı karşılaştırma sonucu: 5 model
      YOLOv11-Nano         mAP50=0.3442
      YOLOv11-Small        mAP50=0.3344
      YOLOv11-Medium       mAP50=0.4325
      YOLOv11-Large        mAP50=0.4279
      YOLOv8-Me

---
## 🔧 HÜCRE 1: Kurulum + Global Ayarlar

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔧 HÜCRE 1: KURULUM + GLOBAL AYARLAR
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NE YAPAR?
#    - Gerekli kütüphaneleri kurar
#    - Tüm proje ayarlarını tek yerde tanımlar
#    - GPU kontrolü yapar
#
# ⚠️ GÜNCELLEME NOTU (v4):
#    - IMG_SIZE 640 → 960 (ince çizikler için)
#    - EPOCHS 60 → 100
#    - KONUM_* ayarları AYRILDI (parçalar büyük nesne, 640 yeterli + OOM önlemi)
#    - MIN_ANNOTATION_ALAN eklendi (küçük annotation filtresi)
#    - ZORLA_YENI_VERSION eklendi (1024'lük veri için)
#    - TILE_* ayarları eklendi (görsel bölme, Hücre 4D)
# ═══════════════════════════════════════════════════════════════════════════════

!pip install ultralytics roboflow reportlab openpyxl shapely -q

import os, torch, warnings
warnings.filterwarnings('ignore')

# ─── ROBOFLOW: KENDİ HASAR VERİN ──────────────────────────────────────────────
API_KEY        = os.environ['ROBOFLOW_API_KEY']   # ⚠️ burhans-workspace-fev5o için API key
WORKSPACE      = 'burhans-workspace-fev5o'
HASAR_PROJECT  = 'my-first-project-2-llolj-0rqmz'

# ⚠️ VERSION SEÇİMİ — ÇOK ÖNEMLİ
#    v1 = 640x640 (eski, düşük çözünürlük — 960 eğitimi anlamsız olur)
#    Yeni = 1024x1024 (ZORLA_YENI_VERSION=True ile oluşturulur)
HASAR_VERSION       = None    # None = en son version otomatik seçilir
ZORLA_YENI_VERSION  = True    # True = 1024'lük yeni version oluştur

# ─── MODEL 1: KONUM (PARÇA) TESPİTİ ───────────────────────────────────────────
KONUM_DATASET = 'carparts-seg.yaml'

KONUM_MAP = {
    'back_bumper': 'Tampon',       'front_bumper': 'Tampon',
    'back_glass':  'Cam',          'front_glass':  'Cam',
    'back_left_door':   'Kapi',    'back_right_door':  'Kapi',
    'front_left_door':  'Kapi',    'front_right_door': 'Kapi',
    'back_left_light':  'Far',     'back_right_light': 'Far',
    'front_left_light': 'Far',     'front_right_light':'Far',
    'hood':        'Kaput',
    'left_mirror': 'Dikiz-Ayna',   'right_mirror': 'Dikiz-Ayna',
    'tailgate':    'Bagaj',        'trunk':        'Bagaj',
    'wheel':       'Camurluk',
}
KONUM_SINIFLAR = ['Tampon', 'Kaput', 'Kapi', 'Camurluk', 'Bagaj', 'Far', 'Cam', 'Dikiz-Ayna']

# ─── MODEL 2: HASAR TİPİ TESPİTİ ──────────────────────────────────────────────
HASAR_MAP = {
    'cizik-hasari': 'Cizik',  'cizik-hasar': 'Cizik',  'cizik': 'Cizik',
    'gocuk-hasari': 'Gocuk',  'gocuk-hasar': 'Gocuk',  'gocuk': 'Gocuk',
    'kirik-hasari': 'Kirik',  'kirik-hasar': 'Kirik',  'kirik': 'Kirik',
}
HASAR_SINIFLAR = ['Cizik', 'Gocuk', 'Kirik']

# ─── TEZ İÇİN: KARŞILAŞTIRILACAK MODELLER ─────────────────────────────────────
# ⚠️ 960 çözünürlük çok VRAM yiyor → batch değerleri düşük
# 💡 ÖNCE SADECE MEDIUM İLE TEST ET
KARSILASTIRMA_MODELLERI = [
    ('yolo11m-seg', 'YOLOv11-Medium', 8),
    # ('yolo11n-seg', 'YOLOv11-Nano',   16),
    # ('yolo11s-seg', 'YOLOv11-Small',  12),
    # ('yolo11l-seg', 'YOLOv11-Large',   6),
    # ('yolov8m-seg', 'YOLOv8-Medium',   8),
]

# ─── MODEL 2 (HASAR) EĞİTİM PARAMETRELERİ ─────────────────────────────────────
# ⚠️ Tiling yapılacaksa veri 4-5x artar → ilk denemede EPOCHS=50 öneriliyor
EPOCHS     = 100      # Tiling ile denemede 50 yap, sonuç iyiyse 100'e çıkar
IMG_SIZE   = 960      # 640 → 960  ⚠️ İnce çizikler için kritik
PATIENCE   = 30       # 20 → 30

# ─── MODEL 1 (KONUM) İÇİN AYRI AYARLAR ────────────────────────────────────────
# ⚠️ Parçalar (tampon, kaput) BÜYÜK nesneler → 960 gereksiz + OOM riski
#    640 ile hem yeterli hem 3 kat hızlı
KONUM_IMG_SIZE = 640
KONUM_EPOCHS   = 60
KONUM_BATCH    = 12

# ─── VERİ KALİTESİ FİLTRESİ (Hücre 4B kullanır) ───────────────────────────────
MIN_ANNOTATION_ALAN = 0.0004   # Görselin %0.04'ünden küçük annotation'lar atılır
                                # Bir sınıfın %40+'ı silinirse 0.0002'ye düşür

# ─── TILE (GÖRSEL BÖLME) AYARLARI — Hücre 4D kullanır ─────────────────────────
# 🎯 AMAÇ: İnce çizikleri büyütmek
#    1024'lük görselde 15px çizik → 2x2 böl → 512'lik tile → YOLO 960'a büyütür
#    Sonuç: çizik ~28px olur (1.9x büyüme) → model artık görebiliyor
#
# ⚠️ VERİ ARTIŞI:
#    (2,2) → veri ~5x artar (4 tile + 1 orijinal)
#    (3,3) → veri ~10x artar (9 tile + 1 orijinal), eğitim çok uzar
TILE_UYGULA     = True        # False → tiling yapma
TILE_GRID       = (2, 2)      # (satır, sütun) — 2x2 = 4 parça
TILE_OVERLAP    = 0.20        # Komşu tile'lar arası %20 örtüşme
ORIJINALI_TUT   = True        # Orijinal görseller de eğitimde kalsın
MIN_POLY_ORAN   = 0.30        # Kırpılan polygon orijinalin %30'undan azsa at
MIN_TILE_ALAN   = 0.0008      # Tile içinde annotation bu orandan küçükse at
BOS_TILE_ORANI  = 0.10        # Hasarsız tile'ların %10'u negatif örnek olarak tutulur

# ─── FÜZYON PARAMETRELERİ ─────────────────────────────────────────────────────
KONUM_CONF   = 0.30
HASAR_CONF   = 0.30
FUZYON_IOU   = 0.10

# NOT: OUTPUT_DIR, RAPOR_DIR, GORSEL_DIR, MODEL_DIZIN, SONUC_JSON → HÜCRE 0'da

# ─── GPU KONTROL ──────────────────────────────────────────────────────────────
print('=' * 70)
print('🔧 KURULUM TAMAMLANDI')
print('=' * 70)
print(f'PyTorch  : {torch.__version__}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU      : {gpu_name} ({gpu_mem:.1f} GB)')
    DEVICE = 0
else:
    print('⚠️  GPU YOK — Eğitim çok yavaş olur!')
    DEVICE = 'cpu'
print()
print(f'Hasar Projesi : {HASAR_PROJECT}')
print(f'Konum Datası  : {KONUM_DATASET} (Ultralytics resmi)')
print(f'Hasar Sınıf   : {HASAR_SINIFLAR}')
print(f'Karşılaştırma : {len(KARSILASTIRMA_MODELLERI)} model')
print()
print('⚙️  MODEL 1 (KONUM) AYARLARI:')
print(f'   Çözünürlük : {KONUM_IMG_SIZE}x{KONUM_IMG_SIZE}')
print(f'   Epoch      : {KONUM_EPOCHS}  |  Batch: {KONUM_BATCH}')
print()
print('⚙️  MODEL 2 (HASAR) AYARLARI:')
print(f'   Çözünürlük : {IMG_SIZE}x{IMG_SIZE}')
print(f'   Epoch      : {EPOCHS}  |  Patience: {PATIENCE}')
print(f'   Min annot. : {MIN_ANNOTATION_ALAN:.4%}')
print()
print('🧩 TILING AYARLARI:')
if TILE_UYGULA:
    _tile_sayisi = TILE_GRID[0] * TILE_GRID[1]
    _carpan = _tile_sayisi + (1 if ORIJINALI_TUT else 0)
    print(f'   Grid       : {TILE_GRID[0]}x{TILE_GRID[1]} = {_tile_sayisi} parça')
    print(f'   Örtüşme    : %{TILE_OVERLAP*100:.0f}')
    print(f'   Orijinal   : {"korunacak" if ORIJINALI_TUT else "silinecek"}')
    print(f'   ⚠️ Veri seti ~{_carpan}x artacak → eğitim süresi de ~{_carpan}x')
    print(f'   💡 Çizik boyutu ~{_tile_sayisi**0.5:.1f}x büyüyecek')
else:
    print('   ⏭️  Tiling KAPALI')
print()
if ZORLA_YENI_VERSION:
    print('🔄 ZORLA_YENI_VERSION = True')
    print('   → Hücre 4, 1024x1024 çözünürlükte YENİ version oluşturacak')
    print('   → Bu, 960 eğitiminin işe yaraması için ŞART')
else:
    print('⚠️  ZORLA_YENI_VERSION = False')
    print('   → Mevcut version kullanılacak (640 ise 960 eğitimi anlamsız olur)')
print('=' * 70)

🔧 KURULUM TAMAMLANDI
PyTorch  : 2.11.0+cu128
GPU      : NVIDIA A100-SXM4-80GB (85.1 GB)

Hasar Projesi : my-first-project-2-llolj-0rqmz
Konum Datası  : carparts-seg.yaml (Ultralytics resmi)
Hasar Sınıf   : ['Cizik', 'Gocuk', 'Kirik']
Karşılaştırma : 1 model

⚙️  MODEL 1 (KONUM) AYARLARI:
   Çözünürlük : 640x640
   Epoch      : 60  |  Batch: 12

⚙️  MODEL 2 (HASAR) AYARLARI:
   Çözünürlük : 960x960
   Epoch      : 100  |  Patience: 30
   Min annot. : 0.0400%

🔄 ZORLA_YENI_VERSION = True
   → Hücre 4, 1024x1024 çözünürlükte YENİ version oluşturacak
   → Bu, 960 eğitiminin işe yaraması için ŞART


---
## 🚙 HÜCRE 2: MODEL 1 — Konum (Parça) Veri Kontrolü

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🚙 HÜCRE 2: KONUM VERİ SETİ KONTROLÜ
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NE YAPAR?
#    - Ultralytics Carparts-Seg veri setini indirir (3833 görsel, 23 sınıf)
#    - Sınıf listesini gösterir
#    - KONUM_MAP eşleşmesini doğrular
#
# 📚 KAYNAK:
#    https://docs.ultralytics.com/datasets/segment/carparts-seg
#    3156 train / 401 valid / 276 test
# ═══════════════════════════════════════════════════════════════════════════════

from ultralytics.utils.downloads import download
from ultralytics.data.utils import check_det_dataset

print('📥 Carparts-Seg veri seti kontrol ediliyor...')
konum_data = check_det_dataset(KONUM_DATASET)

print('\n' + '=' * 70)
print('📊 CARPARTS-SEG VERİ SETİ')
print('=' * 70)
print(f"   Konum   : {konum_data['path']}")
print(f"   Sınıf # : {konum_data['nc']}")

orijinal_siniflar = konum_data['names']
print(f'\n🏷️  ORİJİNAL SINIFLAR → TÜRKÇE EŞLEŞME:')

eslesen = 0
eslesmeyen = []
for idx, cls_name in orijinal_siniflar.items():
    key = cls_name.lower().replace(' ', '_').replace('-', '_')
    if key in KONUM_MAP:
        print(f'   ✅ {cls_name:20s} → {KONUM_MAP[key]}')
        eslesen += 1
    else:
        eslesmeyen.append(cls_name)

if eslesmeyen:
    print(f'\n⏭️  EŞLEŞMEYEN ({len(eslesmeyen)} sınıf — kullanılmayacak):')
    print(f'   {", ".join(eslesmeyen)}')

print(f'\n📈 {eslesen}/{len(orijinal_siniflar)} sınıf eşleşti')
print(f'   Hedef Türkçe sınıflar: {KONUM_SINIFLAR}')
print('=' * 70)

📥 Carparts-Seg veri seti kontrol ediliyor...

📊 CARPARTS-SEG VERİ SETİ
   Konum   : /content/datasets/carparts-seg
   Sınıf # : 23

🏷️  ORİJİNAL SINIFLAR → TÜRKÇE EŞLEŞME:
   ✅ back_bumper          → Tampon
   ✅ back_glass           → Cam
   ✅ back_left_door       → Kapi
   ✅ back_left_light      → Far
   ✅ back_right_door      → Kapi
   ✅ back_right_light     → Far
   ✅ front_bumper         → Tampon
   ✅ front_glass          → Cam
   ✅ front_left_door      → Kapi
   ✅ front_left_light     → Far
   ✅ front_right_door     → Kapi
   ✅ front_right_light    → Far
   ✅ hood                 → Kaput
   ✅ left_mirror          → Dikiz-Ayna
   ✅ right_mirror         → Dikiz-Ayna
   ✅ tailgate             → Bagaj
   ✅ trunk                → Bagaj
   ✅ wheel                → Camurluk

⏭️  EŞLEŞMEYEN (5 sınıf — kullanılmayacak):
   back_door, back_light, front_door, front_light, object

📈 18/23 sınıf eşleşti
   Hedef Türkçe sınıflar: ['Tampon', 'Kaput', 'Kapi', 'Camurluk', 'Bagaj', 'Far', 'Cam', 'D

---
## 🧠 HÜCRE 3: MODEL 1 — Konum Modelini Eğit

Carparts-Seg üzerinde YOLOv11m-seg eğitimi. Bu model tek başına eğitilir, karşılaştırmaya dahil değil (karşılaştırma hasar modeli için).

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🧠 HÜCRE 3: MODEL 1 EĞİTİMİ — KONUM (PARÇA) TESPİTİ
# ═══════════════════════════════════════════════════════════════════════════════
#
# ⚠️ GÜNCELLEME (v3):
#    Artık KONUM_IMG_SIZE (640) / KONUM_EPOCHS (60) / KONUM_BATCH (12) kullanıyor.
#    Önceki halinde IMG_SIZE=960 + batch=12 vardı → CUDA out of memory veriyordu.
#
# 💡 NEDEN 640 YETERLİ?
#    Tampon, kaput, kapı gibi parçalar görselin BÜYÜK bir kısmını kaplar.
#    Yüksek çözünürlük sadece ince çizik/göçük için gerekli (Model 2).
#
# ⏱️ SÜRE: A100'de ~25 dk | V100'de ~40 dk | T4'te ~70 dk
#
# 🎛️ AYAR:
#    EGIT_KONUM = False → mevcut modeli kullan, eğitimi atla
# ═══════════════════════════════════════════════════════════════════════════════

from ultralytics import YOLO
import os

EGIT_KONUM = True   # False yaparsan eğitimi atlar

KONUM_MODEL_PATH = os.path.join(MODEL_DIZIN, 'konum', 'weights', 'best.pt')

if EGIT_KONUM or not os.path.exists(KONUM_MODEL_PATH):
    print('=' * 70)
    print('🚀 MODEL 1 (KONUM) EĞİTİMİ')
    print('=' * 70)
    print(f'   Veri       : {KONUM_DATASET}')
    print(f'   Model      : yolo11m-seg')
    print(f'   Çözünürlük : {KONUM_IMG_SIZE}x{KONUM_IMG_SIZE}')
    print(f'   Epoch      : {KONUM_EPOCHS}')
    print(f'   Batch      : {KONUM_BATCH}')
    print(f'   Kayıt yeri : {MODEL_DIZIN}/konum')
    print('=' * 70)

    konum_model = YOLO('yolo11m-seg.pt')
    konum_model.train(
        data=KONUM_DATASET,
        epochs=KONUM_EPOCHS,        # ← EPOCHS değil, KONUM_EPOCHS (60)
        imgsz=KONUM_IMG_SIZE,       # ← IMG_SIZE değil, KONUM_IMG_SIZE (640)
        batch=KONUM_BATCH,          # ← sabit 12 değil, değişken
        device=DEVICE,
        project=MODEL_DIZIN,
        name='konum',
        exist_ok=True,
        pretrained=True,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        patience=20,
        workers=4,
        cache=False,
        # ─── AUGMENTATION ──────────────────────────────────────────────────
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        fliplr=0.5, flipud=0.0,
        mosaic=1.0, close_mosaic=10,
        scale=0.5, translate=0.1,
        plots=True,
        verbose=True,
    )

    print('\n' + '=' * 70)
    print('✅ MODEL 1 TAMAMLANDI')
    print('=' * 70)
    print(f'   Model: {KONUM_MODEL_PATH}')

    # Hızlı doğrulama
    if os.path.exists(KONUM_MODEL_PATH):
        boyut = os.path.getsize(KONUM_MODEL_PATH) / 1e6
        print(f'   Boyut: {boyut:.1f} MB')
    else:
        print('   ⚠️ Model dosyası bulunamadı, eğitim başarısız olmuş olabilir.')
else:
    print(f'⏭️  Eğitim atlandı. Mevcut model kullanılacak:')
    print(f'   {KONUM_MODEL_PATH}')

🚀 MODEL 1 (KONUM) EĞİTİMİ
   Veri       : carparts-seg.yaml
   Model      : yolo11m-seg
   Çözünürlük : 640x640
   Epoch      : 60
   Batch      : 12
   Kayıt yeri : /content/drive/MyDrive/tez_arac_hasar/modeller/konum
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=carparts-seg.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640,

KeyboardInterrupt: 

---
## 📥 HÜCRE 4: MODEL 2 — Kendi Hasar Verini İndir + Temizle

Bu hücre veriyi indirir, 3 hedef sınıfa filtreler, bozuk/boş kayıtları temizler ve split kontrolü yapar.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📥 HÜCRE 4: HASAR VERİSİ — İNDİR + FİLTRELE + TEMİZLE
# ═══════════════════════════════════════════════════════════════════════════════
#
# ⚠️ GÜNCELLEME (v3):
#    - ZORLA_YENI_VERSION desteği eklendi
#    - Resize 640 → 1024, "Stretch to" → "Fit within" (en-boy oranı korunur)
#    - İndirilen görsellerin gerçek boyutu kontrol ediliyor
#
# 🎯 NEDEN 1024?
#    IMG_SIZE=960 ile eğitim yapacağız. Eğer veri 640'ta kaydedilmişse
#    YOLO onu 960'a BÜYÜTÜR — kaybolan detay geri gelmez, çözünürlük
#    artışının hiçbir faydası olmaz. Kaynak veri 1024 olmalı.
# ═══════════════════════════════════════════════════════════════════════════════

from roboflow import Roboflow
from pathlib import Path
from PIL import Image
import os, yaml, glob, shutil, random, time

# ══ ADIM 1: VERSION HAZIRLA ════════════════════════════════════════════════════
rf = Roboflow(api_key=API_KEY)
proj = rf.workspace(WORKSPACE).project(HASAR_PROJECT)

mevcut_versions = proj.versions()
print(f'📦 Projede {len(mevcut_versions)} version var.')

# Yeni version gerekli mi?
yeni_gerekli = ZORLA_YENI_VERSION or not mevcut_versions

if yeni_gerekli:
    if ZORLA_YENI_VERSION and mevcut_versions:
        print('🔄 ZORLA_YENI_VERSION=True → 1024\'lük yeni version oluşturuluyor')
    else:
        print('⚠️  Version yok → yeni oluşturuluyor')

    print('   Ayarlar: Auto-Orient + Resize 1024x1024 (Fit within)')
    print('   Augmentation: YOK (YOLO kendi yapacak)\n')

    onceki_sayi = len(mevcut_versions)

    try:
        yeni_v = proj.generate_version(settings={
            "augmentation": {},
            "preprocessing": {
                "auto-orient": True,
                "resize": {"width": 1024, "height": 1024, "format": "Fit within"}
            }
        })
        print(f'✅ Version oluşturma isteği gönderildi: v{yeni_v}')
    except Exception as e:
        print(f'❌ Otomatik oluşturma başarısız: {str(e)[:180]}')
        print('\n   MANUEL YAP:')
        print(f'   https://app.roboflow.com/{WORKSPACE}/{HASAR_PROJECT}/generate')
        print('   → Preprocessing: Auto-Orient + Resize "Fit within" 1024x1024')
        print('   → Augmentation: hiçbiri')
        print('   → Generate\n')
        raise RuntimeError('Version oluşturulamadı.')

    # Hazır olmasını bekle
    print('\n⏳ Version hazırlanıyor (7000 görsel için 3-10 dk sürebilir)...')
    for deneme in range(90):          # max 15 dk
        time.sleep(10)
        versions = proj.versions()
        if len(versions) > onceki_sayi:
            print(f'\n✅ Yeni version hazır!')
            break
        if deneme % 6 == 0 and deneme > 0:
            print(f'   {deneme*10}s geçti...')
    else:
        print('\n⚠️  Zaman aşımı. Mevcut versionlarla devam ediliyor.')
        versions = proj.versions()
else:
    versions = mevcut_versions
    print('⏭️  Mevcut version kullanılıyor.')

# ── VERSION SEÇ ───────────────────────────────────────────────────────────────
if HASAR_VERSION is None:
    HASAR_VERSION = int(str(versions[0].version).split('/')[-1])

print(f'\n📥 İndiriliyor: {HASAR_PROJECT} v{HASAR_VERSION}')
dataset = proj.version(HASAR_VERSION).download('yolov11')
DATASET_DIR = dataset.location
print(f'✅ {DATASET_DIR}')

# ══ ÇÖZÜNÜRLÜK DOĞRULAMASI (KRİTİK) ════════════════════════════════════════════
print('\n🔍 İndirilen görsellerin çözünürlüğü kontrol ediliyor...')
ornek_gorseller = []
for sp in ['train', 'valid', 'test']:
    d = os.path.join(DATASET_DIR, sp, 'images')
    if os.path.exists(d):
        ornek_gorseller.extend(list(Path(d).glob('*.*'))[:5])
    if len(ornek_gorseller) >= 5:
        break

if ornek_gorseller:
    boyutlar = []
    for gp in ornek_gorseller[:5]:
        try:
            with Image.open(gp) as im:
                boyutlar.append(max(im.size))
        except Exception:
            pass
    if boyutlar:
        ort_boyut = sum(boyutlar) / len(boyutlar)
        print(f'   Örnek görsel boyutları (uzun kenar): {boyutlar}')
        print(f'   Ortalama: {ort_boyut:.0f} px')

        if ort_boyut < IMG_SIZE * 0.85:
            print(f'\n   ⚠️⚠️  UYARI: Görseller {ort_boyut:.0f}px, eğitim {IMG_SIZE}px!')
            print(f'   YOLO görselleri BÜYÜTECEK — çözünürlük artışının faydası olmaz.')
            print(f'   ÖNERİ: Ya yeni version oluştur (1024), ya IMG_SIZE\'ı 640 yap.')
        else:
            print(f'   ✅ Çözünürlük uygun (eğitim {IMG_SIZE}px ile uyumlu)')

# ══ ADIM 2: SINIF EŞLEŞTİR ═════════════════════════════════════════════════════
data_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
with open(data_yaml_path) as f:
    orig = yaml.safe_load(f)
orig_names = orig['names']

print(f'\n📋 Orijinal sınıflar ({len(orig_names)}):')
OLD_TO_NEW = {}
for old_idx, cls in enumerate(orig_names):
    key = cls.lower().strip()
    if key in HASAR_MAP:
        new_cls = HASAR_MAP[key]
        new_idx = HASAR_SINIFLAR.index(new_cls)
        OLD_TO_NEW[old_idx] = new_idx
        print(f'   ✅ {cls:25s} → {new_cls} (idx {new_idx})')
    else:
        print(f'   ⏭️  {cls:25s} → atla')

if not OLD_TO_NEW:
    print(f'\n❌ Hiçbir sınıf eşleşmedi!')
    print(f'   Roboflow sınıfları: {orig_names}')
    print(f'   HASAR_MAP anahtarları: {list(HASAR_MAP.keys())}')
    raise RuntimeError('Sınıf eşleşmesi başarısız — HASAR_MAP güncelle.')

# ══ ADIM 3+4: FİLTRELE + TEMİZLE ═══════════════════════════════════════════════
print(f'\n🧹 Filtreleme ve temizlik...')

for c in glob.glob(f'{DATASET_DIR}/**/*.cache', recursive=True):
    os.remove(c)

sinif_sayaci = {s: 0 for s in HASAR_SINIFLAR}
toplam_silinen = 0

for split in ['train', 'valid', 'test']:
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    if not os.path.exists(lbl_dir):
        continue

    silinen = 0
    for lf in list(Path(lbl_dir).glob('*.txt')):
        with open(lf) as f:
            lines = f.readlines()

        gecerli = []
        for line in lines:
            p = line.strip().split()
            if not p:
                continue
            try:
                old_id = int(p[0])
            except ValueError:
                continue
            if old_id not in OLD_TO_NEW:
                continue

            coords = p[1:]
            n = len(coords)
            if n < 6 or n % 2 != 0:      # Polygon: min 3 nokta, çift koordinat
                continue
            try:
                vals = [float(x) for x in coords]
                if not all(0.0 <= v <= 1.0 for v in vals):
                    continue
            except ValueError:
                continue

            new_id = OLD_TO_NEW[old_id]
            gecerli.append(f'{new_id} ' + ' '.join(coords))
            sinif_sayaci[HASAR_SINIFLAR[new_id]] += 1

        if gecerli:
            with open(lf, 'w') as f:
                f.write('\n'.join(gecerli) + '\n')
        else:
            for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
                ip = os.path.join(img_dir, lf.stem + ext)
                if os.path.exists(ip):
                    os.remove(ip)
                    break
            os.remove(lf)
            silinen += 1

    # Eşleşmeyen görselleri sil
    lbl_stems = {p.stem for p in Path(lbl_dir).glob('*.txt')}
    for ip in Path(img_dir).glob('*.*'):
        if ip.stem not in lbl_stems:
            os.remove(ip)

    toplam_silinen += silinen
    kalan = len(list(Path(lbl_dir).glob('*.txt')))
    print(f'   {split:6s}: {silinen:5d} silindi → {kalan:5d} kaldı')

# ══ ADIM 5: SPLIT KONTROLÜ ═════════════════════════════════════════════════════
tr_img = os.path.join(DATASET_DIR, 'train', 'images')
tr_lbl = os.path.join(DATASET_DIR, 'train', 'labels')
va_img = os.path.join(DATASET_DIR, 'valid', 'images')
va_lbl = os.path.join(DATASET_DIR, 'valid', 'labels')
te_img = os.path.join(DATASET_DIR, 'test',  'images')
te_lbl = os.path.join(DATASET_DIR, 'test',  'labels')

def gorsel_say(d):
    return len(list(Path(d).glob('*.*'))) if os.path.exists(d) else 0

if gorsel_say(va_img) == 0 or gorsel_say(te_img) == 0:
    print(f'\n🔀 Valid/test yok → train\'den ayrılıyor (70/20/10)...')
    for d in [va_img, va_lbl, te_img, te_lbl]:
        os.makedirs(d, exist_ok=True)

    imgs = sorted(Path(tr_img).glob('*.*'))
    random.seed(42)
    random.shuffle(imgs)
    n = len(imgs)
    n_va, n_te = int(n * 0.20), int(n * 0.10)

    for img_list, (di, dl) in [(imgs[:n_va], (va_img, va_lbl)),
                                (imgs[n_va:n_va+n_te], (te_img, te_lbl))]:
        for ip in img_list:
            shutil.move(str(ip), os.path.join(di, ip.name))
            lp = os.path.join(tr_lbl, ip.stem + '.txt')
            if os.path.exists(lp):
                shutil.move(lp, os.path.join(dl, ip.stem + '.txt'))

# ══ data.yaml YAZ ══════════════════════════════════════════════════════════════
with open(data_yaml_path, 'w') as f:
    yaml.dump({
        'train': tr_img, 'val': va_img, 'test': te_img,
        'nc': len(HASAR_SINIFLAR), 'names': HASAR_SINIFLAR,
    }, f, default_flow_style=False, sort_keys=False)

HASAR_DATA_YAML = data_yaml_path

# ══ RAPOR ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('✅ HASAR VERİSİ HAZIR')
print('=' * 70)
print(f'   Version: v{HASAR_VERSION}')
print(f'   Train  : {gorsel_say(tr_img):5d} görsel')
print(f'   Valid  : {gorsel_say(va_img):5d} görsel')
print(f'   Test   : {gorsel_say(te_img):5d} görsel')
print(f'   ─────────────────────')
print(f'   TOPLAM : {gorsel_say(tr_img)+gorsel_say(va_img)+gorsel_say(te_img):5d} görsel')

print(f'\n🏷️  ANNOTATION DAĞILIMI:')
tot = sum(sinif_sayaci.values())
for s, c in sorted(sinif_sayaci.items(), key=lambda x: -x[1]):
    pct = c / tot * 100 if tot else 0
    bar = '█' * int(pct / 3)
    durum = '✅' if c >= 1500 else ('⚠️ ' if c >= 500 else '❌')
    print(f'   {durum} {s:8s}: {c:6d} ({pct:5.1f}%) {bar}')
print(f'\n   Toplam annotation: {tot}')
print(f'   Silinen (boş/geçersiz): {toplam_silinen}')
print(f'\n   data.yaml: {HASAR_DATA_YAML}')
print('=' * 70)
print('\n➡️  SONRAKİ: Hücre 4A (harici veri) → 4B (filtre) → 4C (split) → 5 (eğitim)')

🔧 DEĞİŞİKLİK 2 — YENİ Hücre 4A: Harici Veri Ekle (CarDD)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🌐 HÜCRE 4A: HARİCİ VERİ SETİ BİRLEŞTİRME (Ablation Study İçin)
# ═══════════════════════════════════════════════════════════════════════════════
#
# ⚠️ GÜNCELLEME (v3):
#    - FORMAT KONTROLÜ eklendi (bbox mı polygon mu?)
#      Önceki halinde bbox verisi sessizce eleniyordu, sebebi anlaşılmıyordu.
#
# 🎯 NE YAPAR?
#    Roboflow Universe'den açık kaynak hasar veri setini indirir,
#    sınıflarını bizimkilere map'ler ve mevcut veriye EKLER.
#
# 📚 TEZ İÇİN:
#    Ablation study: "Sadece kendi veri" vs "Kendi veri + harici"
#    Tezde MUTLAKA atıf ver ve birleşimi açıkça belirt.
#
# 🎛️ AYAR:
#    HARICI_VERI_EKLE = False → atla (sadece kendi verin)
# ═══════════════════════════════════════════════════════════════════════════════

import os

HARICI_VERI_EKLE = True

# Aday veri setleri (segmentation formatında olmalı)
HARICI_KAYNAKLAR = [
    ('sinfo', 'car-damage-segmentation', None),   # 4303 görsel, seg
    # ('dan-vmm5z', 'car-damage-coco-dataset', None),  # 1700 görsel, alternatif
]

# Harici sınıf → Bizim sınıf eşlemesi (küçük harf, tire normalize)
HARICI_MAP = {
    'scratch': 'Cizik',   'scratches': 'Cizik',  'cizik': 'Cizik',
    'minor-scratches': 'Cizik', 'scratch-damage': 'Cizik',
    'dent': 'Gocuk',      'dents': 'Gocuk',      'gocuk': 'Gocuk',
    'dent-damage': 'Gocuk', 'deformation': 'Gocuk',
    'crack': 'Kirik',     'cracks': 'Kirik',     'kirik': 'Kirik',
    'glass-shatter': 'Kirik', 'glass shatter': 'Kirik',
    'broken': 'Kirik',    'lamp-broken': 'Kirik', 'shatter': 'Kirik',
}

if not HARICI_VERI_EKLE:
    print('⏭️  Harici veri eklenmiyor (HARICI_VERI_EKLE = False)')
    ABLATION_ETIKET = 'sadece_kendi_veri'
else:
    import shutil, yaml, glob
    from pathlib import Path
    from roboflow import Roboflow

    ABLATION_ETIKET = 'kendi_veri_plus_harici'
    rf_h = Roboflow(api_key=API_KEY)

    toplam_eklenen = 0
    eklenen_sinif = {s: 0 for s in HASAR_SINIFLAR}

    for ws, prj, ver in HARICI_KAYNAKLAR:
        print(f'\n{"═"*70}')
        print(f'🌐 İNDİRİLİYOR: {ws}/{prj}')
        print(f'{"═"*70}')

        try:
            hp = rf_h.workspace(ws).project(prj)
            vlist = hp.versions()
            if not vlist:
                print(f'   ⚠️ Version yok, atlanıyor.')
                continue
            v = ver or int(str(vlist[0].version).split('/')[-1])
            hd = hp.version(v).download('yolov11', location=f'/content/_harici_{prj}')
            HDIR = hd.location
            print(f'   ✅ İndirildi: {HDIR}')
        except Exception as e:
            print(f'   ❌ İndirme hatası: {str(e)[:180]}')
            continue

        # ─── SINIFLARI OKU ────────────────────────────────────────────────
        try:
            with open(os.path.join(HDIR, 'data.yaml')) as f:
                hnames = yaml.safe_load(f)['names']
        except Exception as e:
            print(f'   ❌ data.yaml okunamadı: {str(e)[:100]}')
            shutil.rmtree(HDIR, ignore_errors=True)
            continue

        print(f'\n   📋 Harici sınıflar ({len(hnames)}):')
        H_OLD_TO_NEW = {}
        for i, cn in enumerate(hnames):
            k = cn.lower().strip().replace('_', '-')
            if k in HARICI_MAP:
                yeni = HARICI_MAP[k]
                H_OLD_TO_NEW[i] = HASAR_SINIFLAR.index(yeni)
                print(f'      ✅ {cn:25s} → {yeni}')
            else:
                print(f'      ⏭️  {cn:25s} → atla')

        if not H_OLD_TO_NEW:
            print(f'\n   ⚠️ Hiçbir sınıf eşleşmedi. HARICI_MAP\'e şunları ekle:')
            print(f'      {hnames}')
            shutil.rmtree(HDIR, ignore_errors=True)
            continue

        # ─── FORMAT KONTROLÜ: BBOX MI POLYGON MU? (YENİ) ──────────────────
        print(f'\n   🔍 Annotation formatı kontrol ediliyor...')
        kontrol_dir = None
        for hs in ['train', 'valid', 'test']:
            d = os.path.join(HDIR, hs, 'labels')
            if os.path.exists(d) and list(Path(d).glob('*.txt')):
                kontrol_dir = d
                break

        if kontrol_dir is None:
            print(f'   ❌ Label klasörü bulunamadı — atlanıyor.')
            shutil.rmtree(HDIR, ignore_errors=True)
            continue

        ornek = list(Path(kontrol_dir).glob('*.txt'))[:30]
        poly_sayisi, bbox_sayisi = 0, 0
        for of in ornek:
            with open(of) as f:
                for line in f:
                    pp = line.strip().split()
                    if len(pp) == 5:
                        bbox_sayisi += 1
                    elif len(pp) >= 7:
                        poly_sayisi += 1

        print(f'      Polygon satır: {poly_sayisi} | BBox satır: {bbox_sayisi}')

        if poly_sayisi == 0:
            print(f'   ❌ Bu veri seti BBOX formatında (polygon yok).')
            print(f'      Segmentation eğitimi için KULLANILAMAZ — atlanıyor.')
            print(f'      💡 Roboflow Universe\'de "Instance Segmentation" etiketli')
            print(f'         bir veri seti seç.')
            shutil.rmtree(HDIR, ignore_errors=True)
            continue

        print(f'   ✅ Polygon formatı doğrulandı.')

        # ─── DOSYALARI AKTAR ──────────────────────────────────────────────
        print(f'\n   📦 Veriler aktarılıyor...')
        hedef_img = os.path.join(DATASET_DIR, 'train', 'images')
        hedef_lbl = os.path.join(DATASET_DIR, 'train', 'labels')
        os.makedirs(hedef_img, exist_ok=True)
        os.makedirs(hedef_lbl, exist_ok=True)

        eklenen = 0
        gorsel_bulunamadi = 0

        for hsplit in ['train', 'valid', 'test']:
            hlbl = os.path.join(HDIR, hsplit, 'labels')
            himg = os.path.join(HDIR, hsplit, 'images')
            if not os.path.exists(hlbl):
                continue

            for lf in Path(hlbl).glob('*.txt'):
                with open(lf) as f:
                    lines = f.readlines()

                gecerli = []
                for line in lines:
                    p = line.strip().split()
                    if not p:
                        continue
                    try:
                        oid = int(p[0])
                    except ValueError:
                        continue
                    if oid not in H_OLD_TO_NEW:
                        continue
                    c = p[1:]
                    if len(c) < 6 or len(c) % 2 != 0:
                        continue
                    try:
                        vals = [float(x) for x in c]
                        if not all(0.0 <= x <= 1.0 for x in vals):
                            continue
                    except ValueError:
                        continue
                    nid = H_OLD_TO_NEW[oid]
                    gecerli.append(f'{nid} ' + ' '.join(c))

                if not gecerli:
                    continue

                # Görseli bul ve kopyala (prefix ile isim çakışması önlenir)
                bulundu = False
                for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
                    src = os.path.join(himg, lf.stem + ext)
                    if os.path.exists(src):
                        yeni_ad = f'HARICI_{prj[:12]}_{lf.stem}'
                        shutil.copy(src, os.path.join(hedef_img, yeni_ad + ext))
                        with open(os.path.join(hedef_lbl, yeni_ad + '.txt'), 'w') as wf:
                            wf.write('\n'.join(gecerli) + '\n')
                        # Sayaçları güncelle
                        for g in gecerli:
                            eklenen_sinif[HASAR_SINIFLAR[int(g.split()[0])]] += 1
                        bulundu = True
                        eklenen += 1
                        break

                if not bulundu:
                    gorsel_bulunamadi += 1

        toplam_eklenen += eklenen
        print(f'   ✅ {eklenen} görsel eklendi')
        if gorsel_bulunamadi:
            print(f'   ⚠️ {gorsel_bulunamadi} label\'ın görseli bulunamadı (atlandı)')

        shutil.rmtree(HDIR, ignore_errors=True)

    # Cache temizle
    for c in glob.glob(f'{DATASET_DIR}/**/*.cache', recursive=True):
        os.remove(c)

    # ─── RAPOR ────────────────────────────────────────────────────────────────
    print('\n' + '=' * 70)
    print('✅ HARİCİ VERİ BİRLEŞTİRME TAMAMLANDI')
    print('=' * 70)
    print(f'   Eklenen görsel: {toplam_eklenen}')

    if toplam_eklenen == 0:
        print('\n   ⚠️ HİÇ VERİ EKLENMEDİ!')
        print('   Olası sebepler:')
        print('     - Veri seti bbox formatında (polygon değil)')
        print('     - Sınıf isimleri HARICI_MAP ile eşleşmedi')
        print('     - İndirme başarısız oldu')
        print('\n   💡 HARICI_VERI_EKLE = False yapıp devam edebilirsin.')
        ABLATION_ETIKET = 'sadece_kendi_veri'
    else:
        print(f'\n   Eklenen annotation:')
        for s, c in eklenen_sinif.items():
            print(f'      {s:8s}: {c}')

        tr_n = len(list(Path(os.path.join(DATASET_DIR, "train", "images")).glob("*.*")))
        print(f'\n   Train klasöründe toplam: {tr_n} görsel')
        print('\n   📝 TEZ NOTU: Veri seti birleşimini metodoloji bölümünde belirt')
        print('      ve kaynak veri setine atıf ver.')

    print(f'\n   Ablation etiketi: {ABLATION_ETIKET}')
    print('=' * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🧩 HÜCRE 4D: TILE-BASED VERİ ARTIRMA (İnce Çizikler İçin)
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 PROBLEM:
#    1024x1024 bir görselde 15 piksellik çizik → %1.5 alan kaplar.
#    YOLO 960'a küçültünce ~14 piksel kalır. Hala çok küçük.
#
# 💡 ÇÖZÜM (Tiling):
#    Görseli 2x2 parçaya böl → her parça 512x512 olur
#    YOLO onu 960'a BÜYÜTÜR → çizik 14px yerine ~26px olur (1.9x büyüme)
#    Model artık çiziği görebiliyor!
#
# 📐 NASIL ÇALIŞIR?
#    ┌─────────┬─────────┐   Her tile ayrı bir eğitim görseli olur.
#    │ Tile 1  │ Tile 2  │   Polygon'lar tile koordinatlarına dönüştürülür.
#    ├─────────┼─────────┤   Tile sınırında kesilen polygon'lar KIRPILIR.
#    │ Tile 3  │ Tile 4  │   %20 örtüşme → sınırdaki hasarlar kaybolmaz.
#    └─────────┴─────────┘
#
# 📚 TEZ İÇİN:
#    Bu literatürde "tiling" / "sliced inference" olarak geçer (SAHI, 2022).
#    Küçük nesne tespitinde standart bir tekniktir — özgün katkı olarak yazılabilir.
#
# 🎛️ AYARLAR:
#    TILE_UYGULA    : False → atla
#    TILE_GRID      : (2,2) = 4 parça | (3,3) = 9 parça (daha agresif)
#    TILE_OVERLAP   : Komşu tile'lar arası örtüşme oranı
#    ORIJINALI_TUT  : True → orijinal görseller de kalsın (önerilen)
#    MIN_POLY_ORAN  : Kırpılan polygon orijinalin %kaçı kalmalı ki tutulsun
# ═══════════════════════════════════════════════════════════════════════════════
# Ayarlar HÜCRE 1'de tanımlı. Burada değiştirmek istersen üzerine yaz:
# TILE_GRID = (3, 3)   # daha agresif bölme

# Güvenlik: Hücre 1 çalıştırılmamışsa varsayılan ata
for _var, _val in [('TILE_UYGULA', True), ('TILE_GRID', (2, 2)),
                   ('TILE_OVERLAP', 0.20), ('ORIJINALI_TUT', True),
                   ('MIN_POLY_ORAN', 0.30), ('MIN_TILE_ALAN', 0.0008),
                   ('BOS_TILE_ORANI', 0.10)]:
    if _var not in globals():
        globals()[_var] = _val
        print(f'⚠️  {_var} tanımsızdı → varsayılan {_val} atandı')

if not TILE_UYGULA:
    print('⏭️  Tiling atlandı (TILE_UYGULA = False)')
else:
    !pip install shapely -q

    import os, shutil, random, glob
    import numpy as np
    from pathlib import Path
    from PIL import Image
    from shapely.geometry import Polygon, box as shapely_box
    from shapely.validation import make_valid
    from collections import Counter

    Image.MAX_IMAGE_PIXELS = None
    random.seed(42)

    print('=' * 70)
    print('🧩 TILE-BASED VERİ ARTIRMA')
    print('=' * 70)
    print(f'   Grid        : {TILE_GRID[0]}x{TILE_GRID[1]} = {TILE_GRID[0]*TILE_GRID[1]} parça')
    print(f'   Örtüşme     : %{TILE_OVERLAP*100:.0f}')
    print(f'   Orijinali tut: {ORIJINALI_TUT}')
    print('=' * 70)

    # ── ÇALIŞMA KLASÖRÜ ───────────────────────────────────────────────────────
    TILE_OUT = '/content/_tiled'
    shutil.rmtree(TILE_OUT, ignore_errors=True)
    os.makedirs(f'{TILE_OUT}/images', exist_ok=True)
    os.makedirs(f'{TILE_OUT}/labels', exist_ok=True)

    # ── YARDIMCI: POLYGON KIRPMA ──────────────────────────────────────────────
    def polygon_kirp(norm_coords, tile_x0, tile_y0, tile_w, tile_h, W, H):
        """Normalize polygon'u tile'a kırpar, tile-normalize koordinat döner.

        norm_coords : [x1,y1,x2,y2,...] görsel-normalize (0-1)
        tile_*      : tile'ın piksel koordinatları
        W, H        : orijinal görsel boyutu
        Döner       : [(kirpilmis_coords, kalan_oran), ...] veya []
        """
        # Normalize → piksel
        pts = [(norm_coords[i] * W, norm_coords[i+1] * H)
               for i in range(0, len(norm_coords), 2)]
        if len(pts) < 3:
            return []

        try:
            poly = Polygon(pts)
            if not poly.is_valid:
                poly = make_valid(poly)
            if poly.is_empty or poly.area <= 0:
                return []
        except Exception:
            return []

        orijinal_alan = poly.area
        tile_kutu = shapely_box(tile_x0, tile_y0, tile_x0 + tile_w, tile_y0 + tile_h)

        try:
            kesisim = poly.intersection(tile_kutu)
        except Exception:
            return []

        if kesisim.is_empty or kesisim.area <= 0:
            return []

        # MultiPolygon olabilir → parçalara ayır
        parcalar = []
        geoms = [kesisim] if kesisim.geom_type == 'Polygon' else list(getattr(kesisim, 'geoms', []))

        for g in geoms:
            if g.geom_type != 'Polygon' or g.area <= 0:
                continue
            kalan_oran = g.area / orijinal_alan
            if kalan_oran < MIN_POLY_ORAN:
                continue

            # Tile-normalize koordinatlara çevir
            ext = list(g.exterior.coords)[:-1]   # son nokta ilkiyle aynı
            if len(ext) < 3:
                continue

            yeni = []
            for px, py in ext:
                nx = (px - tile_x0) / tile_w
                ny = (py - tile_y0) / tile_h
                nx = min(max(nx, 0.0), 1.0)
                ny = min(max(ny, 0.0), 1.0)
                yeni.extend([nx, ny])

            # Tile içindeki alan oranı (çok küçükse at)
            tile_alan_orani = g.area / (tile_w * tile_h)
            if tile_alan_orani < MIN_TILE_ALAN:
                continue

            parcalar.append((yeni, kalan_oran))

        return parcalar

    # ── ANA DÖNGÜ ─────────────────────────────────────────────────────────────
    R, C = TILE_GRID
    uretilen_tile   = 0
    bos_tile_tutulan = 0
    atlanan_gorsel  = 0
    tile_sinif      = Counter()
    islenen         = 0

    # Tüm split'lerdeki görselleri topla
    tum_gorseller = []
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(DATASET_DIR, split, 'images')
        lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
        if not os.path.exists(img_dir):
            continue
        for ip in Path(img_dir).glob('*.*'):
            lp = os.path.join(lbl_dir, ip.stem + '.txt')
            if os.path.exists(lp):
                tum_gorseller.append((ip, lp))

    print(f'\n📦 İşlenecek: {len(tum_gorseller)} görsel')
    print(f'   Beklenen tile: ~{len(tum_gorseller) * R * C}\n')

    for ip, lp in tum_gorseller:
        try:
            with Image.open(ip) as im:
                im = im.convert('RGB')
                W, H = im.size

                # Tile boyutları (örtüşmeli)
                tile_w = int(W / C * (1 + TILE_OVERLAP))
                tile_h = int(H / R * (1 + TILE_OVERLAP))
                adim_x = int(W / C)
                adim_y = int(H / R)

                # Etiketleri oku
                with open(lp) as f:
                    satirlar = [s.strip().split() for s in f if s.strip()]

                for r in range(R):
                    for c in range(C):
                        x0 = min(c * adim_x, W - tile_w) if tile_w < W else 0
                        y0 = min(r * adim_y, H - tile_h) if tile_h < H else 0
                        x0, y0 = max(x0, 0), max(y0, 0)
                        tw = min(tile_w, W - x0)
                        th = min(tile_h, H - y0)

                        if tw < 50 or th < 50:
                            continue

                        # Bu tile'daki etiketleri hesapla
                        tile_etiketler = []
                        for p in satirlar:
                            try:
                                cid = int(p[0])
                                coords = [float(x) for x in p[1:]]
                            except (ValueError, IndexError):
                                continue
                            if len(coords) < 6 or len(coords) % 2 != 0:
                                continue

                            parcalar = polygon_kirp(coords, x0, y0, tw, th, W, H)
                            for yeni_coords, _ in parcalar:
                                tile_etiketler.append(
                                    f'{cid} ' + ' '.join(f'{v:.6f}' for v in yeni_coords))
                                tile_sinif[HASAR_SINIFLAR[cid]] += 1

                        # Boş tile'ları kısmen tut (negatif örnek)
                        if not tile_etiketler:
                            if random.random() > BOS_TILE_ORANI:
                                continue
                            bos_tile_tutulan += 1

                        # Tile'ı kes ve kaydet
                        tile_img = im.crop((x0, y0, x0 + tw, y0 + th))
                        tile_ad = f'{ip.stem}_t{r}{c}'
                        tile_img.save(f'{TILE_OUT}/images/{tile_ad}.jpg',
                                      'JPEG', quality=92)
                        with open(f'{TILE_OUT}/labels/{tile_ad}.txt', 'w') as f:
                            f.write('\n'.join(tile_etiketler) + ('\n' if tile_etiketler else ''))
                        uretilen_tile += 1

                islenen += 1
                if islenen % 500 == 0:
                    print(f'   {islenen}/{len(tum_gorseller)} işlendi → {uretilen_tile} tile')

        except Exception as e:
            atlanan_gorsel += 1
            if atlanan_gorsel <= 3:
                print(f'   ⚠️ {ip.name}: {str(e)[:80]}')

    print(f'\n✅ {uretilen_tile} tile üretildi')

    # ── TILE'LARI ANA VERİ SETİNE EKLE ────────────────────────────────────────
    print(f'\n📥 Tile\'lar veri setine ekleniyor...')

    hedef_img = os.path.join(DATASET_DIR, 'train', 'images')
    hedef_lbl = os.path.join(DATASET_DIR, 'train', 'labels')

    # Orijinalleri silmek istiyorsak önce temizle
    if not ORIJINALI_TUT:
        print('   🗑️  Orijinal görseller siliniyor (ORIJINALI_TUT=False)...')
        for split in ['train', 'valid', 'test']:
            for alt in ['images', 'labels']:
                d = os.path.join(DATASET_DIR, split, alt)
                shutil.rmtree(d, ignore_errors=True)
                os.makedirs(d, exist_ok=True)

    os.makedirs(hedef_img, exist_ok=True)
    os.makedirs(hedef_lbl, exist_ok=True)

    tasinan = 0
    for tp in Path(f'{TILE_OUT}/images').glob('*.jpg'):
        lp_src = f'{TILE_OUT}/labels/{tp.stem}.txt'
        if not os.path.exists(lp_src):
            continue
        shutil.move(str(tp), os.path.join(hedef_img, tp.name))
        shutil.move(lp_src, os.path.join(hedef_lbl, tp.stem + '.txt'))
        tasinan += 1

    shutil.rmtree(TILE_OUT, ignore_errors=True)

    # Cache temizle
    for c in glob.glob(f'{DATASET_DIR}/**/*.cache', recursive=True):
        os.remove(c)

    # ── RAPOR ─────────────────────────────────────────────────────────────────
    def say(d):
        return len(list(Path(d).glob('*.*'))) if os.path.exists(d) else 0

    tr_n = say(os.path.join(DATASET_DIR, 'train', 'images'))
    va_n = say(os.path.join(DATASET_DIR, 'valid', 'images'))
    te_n = say(os.path.join(DATASET_DIR, 'test',  'images'))

    print('\n' + '=' * 70)
    print('✅ TILING TAMAMLANDI')
    print('=' * 70)
    print(f'   İşlenen orijinal görsel : {islenen}')
    print(f'   Üretilen tile           : {uretilen_tile}')
    print(f'   Eklenen (taşınan)       : {tasinan}')
    print(f'   Boş tile (negatif örnek): {bos_tile_tutulan}')
    print(f'   Atlanan (hatalı)        : {atlanan_gorsel}')
    print(f'\n   Tile\'lardaki annotation:')
    for s in HASAR_SINIFLAR:
        print(f'      {s:8s}: {tile_sinif[s]}')
    print(f'\n   📊 GÜNCEL VERİ SETİ:')
    print(f'      Train: {tr_n:6d}')
    print(f'      Valid: {va_n:6d}')
    print(f'      Test : {te_n:6d}')
    print(f'      ─────────────')
    print(f'      TOPLAM: {tr_n+va_n+te_n:6d} görsel')
    print('=' * 70)
    print('\n💡 Çiziklerin göreli boyutu ~{:.1f}x büyüdü.'.format(
        (R * C) ** 0.5))
    print('   Şimdi Hücre 4B (küçük annotation filtresi) → 4C (split) → 5 (eğitim)')

🔧 DEĞİŞİKLİK 3 — YENİ Hücre 4B: Küçük Annotation Filtresi

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔍 HÜCRE 4B: ÇOK KÜÇÜK ANNOTATION'LARI FİLTRELE
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NEDEN GEREKLİ?
#    Çok küçük çizikler (görselin %0.04'ünden küçük) YOLO'nun en ince
#    detection katmanına bile sığmıyor. Model bunları öğrenemez ama
#    loss hesabında ceza alır → BÜYÜK çizikleri de kaçırmaya başlar.
#    Bunları atmak modeli DAHA İYİ yapar, kötü değil.
#
# 📐 HESAP:
#    Polygon alanı Shoelace formülüyle hesaplanır (normalize koordinat).
#    MIN_ANNOTATION_ALAN = 0.0004 → görselin %0.04'ü
#    960x960 için: 921600 * 0.0004 ≈ 370 piksel ≈ 19x19 kutu
# ═══════════════════════════════════════════════════════════════════════════════

import os, glob
from pathlib import Path
from collections import Counter

def polygon_alan(coords):
    """Shoelace formülü ile normalize polygon alanı."""
    xs = coords[0::2]
    ys = coords[1::2]
    n = len(xs)
    if n < 3:
        return 0.0
    a = 0.0
    for i in range(n):
        j = (i + 1) % n
        a += xs[i] * ys[j] - xs[j] * ys[i]
    return abs(a) / 2.0

print('=' * 70)
print(f'🔍 KÜÇÜK ANNOTATION FİLTRESİ (eşik: {MIN_ANNOTATION_ALAN:.4%})')
print('=' * 70)

silinen_sinif = Counter()
kalan_sinif   = Counter()
alan_dagilimi = []
bos_kalan     = 0

for split in ['train', 'valid', 'test']:
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    if not os.path.exists(lbl_dir):
        continue

    s_split = 0
    for lf in list(Path(lbl_dir).glob('*.txt')):
        with open(lf) as f:
            lines = f.readlines()

        tutulan = []
        for line in lines:
            p = line.strip().split()
            if len(p) < 7:
                continue
            cid = int(p[0])
            coords = [float(x) for x in p[1:]]
            alan = polygon_alan(coords)
            alan_dagilimi.append(alan)

            if alan >= MIN_ANNOTATION_ALAN:
                tutulan.append(line.strip())
                kalan_sinif[HASAR_SINIFLAR[cid]] += 1
            else:
                silinen_sinif[HASAR_SINIFLAR[cid]] += 1
                s_split += 1

        if tutulan:
            with open(lf, 'w') as f:
                f.write('\n'.join(tutulan) + '\n')
        else:
            # Tamamen boşaldı → görsel + label sil
            for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
                ip = os.path.join(img_dir, lf.stem + ext)
                if os.path.exists(ip):
                    os.remove(ip)
                    break
            os.remove(lf)
            bos_kalan += 1

    kalan = len(list(Path(lbl_dir).glob('*.txt')))
    print(f'   {split:6s}: {s_split:5d} küçük annotation silindi → {kalan:5d} görsel kaldı')

# Cache temizle
for c in glob.glob(f'{DATASET_DIR}/**/*.cache', recursive=True):
    os.remove(c)

# ─── RAPOR ────────────────────────────────────────────────────────────────────
import numpy as np
print('\n' + '=' * 70)
print('📊 FİLTRELEME SONUCU')
print('=' * 70)
print(f'   Boş kalan (silinen) görsel: {bos_kalan}')
print(f'\n   {"Sınıf":10s} {"Silinen":>10s} {"Kalan":>10s} {"Silinme %":>12s}')
print('   ' + '─' * 46)
for s in HASAR_SINIFLAR:
    sil, kal = silinen_sinif[s], kalan_sinif[s]
    tot = sil + kal
    pct = sil / tot * 100 if tot else 0
    uyari = ' ⚠️' if pct > 40 else ''
    print(f'   {s:10s} {sil:10d} {kal:10d} {pct:11.1f}%{uyari}')

if alan_dagilimi:
    a = np.array(alan_dagilimi)
    print(f'\n   📐 ANNOTATION ALAN İSTATİSTİKLERİ:')
    print(f'      Medyan  : {np.median(a):.5f} ({np.median(a)*100:.3f}%)')
    print(f'      Ortalama: {a.mean():.5f}')
    print(f'      %25\'lik : {np.percentile(a, 25):.5f}')
    print(f'      %10\'luk : {np.percentile(a, 10):.5f}')
    print(f'      Eşik    : {MIN_ANNOTATION_ALAN:.5f}')

print('\n   💡 Eğer bir sınıfın %40\'ından fazlası silindiyse (⚠️),')
print('      MIN_ANNOTATION_ALAN değerini düşürmeyi düşün (Hücre 1).')
print('=' * 70)

🔧 DEĞİŞİKLİK 4 — YENİ Hücre 4C: Split Düzelt

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔀 HÜCRE 4C: SPLIT ORANINI DÜZELT (70/20/10)
# ═══════════════════════════════════════════════════════════════════════════════
# Roboflow'dan gelen split dengesizdi (97/2/1) → valid 102, test 47 görseldi.
# Bu kadar küçük setle yapılan ölçüm güvenilir değil.
# ⚠️ EN SON ÇALIŞTIR — harici veri ve filtreleme bittikten sonra.
# ═══════════════════════════════════════════════════════════════════════════════

import os, shutil, random, yaml, glob
from pathlib import Path

# ── 1. TÜM GÖRSELLERİ HAVUZA TOPLA ────────────────────────────────────────────
havuz = []
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    if not os.path.exists(img_dir):
        continue
    for ip in Path(img_dir).glob('*.*'):
        lp = os.path.join(lbl_dir, ip.stem + '.txt')
        if os.path.exists(lp):
            havuz.append((str(ip), lp))

print(f'📦 Havuz: {len(havuz)} görsel-label çifti')

# ── 2. GEÇİCİ KLASÖRE TAŞI ────────────────────────────────────────────────────
GECICI = '/content/_havuz'
shutil.rmtree(GECICI, ignore_errors=True)
os.makedirs(f'{GECICI}/images', exist_ok=True)
os.makedirs(f'{GECICI}/labels', exist_ok=True)

for ip, lp in havuz:
    shutil.move(ip, f'{GECICI}/images/{os.path.basename(ip)}')
    shutil.move(lp, f'{GECICI}/labels/{os.path.basename(lp)}')

# ── 3. KLASÖRLERİ SIFIRLA ─────────────────────────────────────────────────────
for split in ['train', 'valid', 'test']:
    for alt in ['images', 'labels']:
        d = os.path.join(DATASET_DIR, split, alt)
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)

# ── 4. YENİDEN DAĞIT (70/20/10) ───────────────────────────────────────────────
tum = sorted(Path(f'{GECICI}/images').glob('*.*'))
random.seed(42)
random.shuffle(tum)

n = len(tum)
n_tr, n_va = int(n * 0.70), int(n * 0.20)

for split, liste in [('train', tum[:n_tr]),
                     ('valid', tum[n_tr:n_tr+n_va]),
                     ('test',  tum[n_tr+n_va:])]:
    for ip in liste:
        shutil.move(str(ip), os.path.join(DATASET_DIR, split, 'images', ip.name))
        lp = f'{GECICI}/labels/{ip.stem}.txt'
        if os.path.exists(lp):
            shutil.move(lp, os.path.join(DATASET_DIR, split, 'labels', ip.stem + '.txt'))

shutil.rmtree(GECICI, ignore_errors=True)

# ── 5. CACHE SİL + data.yaml YAZ ──────────────────────────────────────────────
for c in glob.glob(f'{DATASET_DIR}/**/*.cache', recursive=True):
    os.remove(c)

with open(HASAR_DATA_YAML, 'w') as f:
    yaml.dump({
        'train': os.path.join(DATASET_DIR, 'train', 'images'),
        'val':   os.path.join(DATASET_DIR, 'valid', 'images'),
        'test':  os.path.join(DATASET_DIR, 'test',  'images'),
        'nc':    len(HASAR_SINIFLAR),
        'names': HASAR_SINIFLAR,
    }, f, default_flow_style=False, sort_keys=False)

# ── 6. RAPOR ──────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('✅ SPLIT DÜZELTİLDİ')
print('=' * 60)
for split in ['train', 'valid', 'test']:
    ni = len(list(Path(os.path.join(DATASET_DIR, split, 'images')).glob('*.*')))
    print(f'   {split:6s}: {ni:5d} görsel  (%{ni/n*100:.1f})')
print('=' * 60)
print(f'   Toplam: {n} görsel')
print(f'\n   ✅ Şimdi Hücre 5\'i çalıştırabilirsin.')

---
## 🏆 HÜCRE 5: MODEL 2 — 5 MODELLİ KARŞILAŞTIRMALI EĞİTİM

**Tezin en değerli bölümü.** 5 farklı mimari aynı veri ve aynı parametrelerle eğitilir, sonuçlar karşılaştırılır.

⏱️ **Süre:** A100'de ~3 saat | V100'de ~4.5 saat | T4'te ~8 saat

In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🏆 HÜCRE 5: KARŞILAŞTIRMALI EĞİTİM
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NE YAPAR?
#    - KARSILASTIRMA_MODELLERI listesindeki modelleri AYNI veriyle eğitir
#    - Her modelin test seti metriklerini kaydeder
#    - Inference hızını ölçer
#    - Sonuçları JSON'a yazar (kesinti olursa kaybolmaz)
#
# ⚠️ GÜNCELLEME NOTU (v2):
#    - imgsz=960 (Hücre 1'den)
#    - mosaic 0.0 → 0.5 (AÇILDI)
#    - copy_paste=0.3 EKLENDİ (segmentasyonda çok etkili)
#    - degrees=5.0, erasing=0.2 EKLENDİ
#    - cls=0.6 (0.5'ten artırıldı, sınıf ayrımı için)
#    - ABLATION_ETIKET model isimlerine eklendi
#
# 💡 KESİNTİ OLURSA:
#    ATLANACAK listesine biten modelleri ekle, tekrar çalıştır.
# ═══════════════════════════════════════════════════════════════════════════════

from ultralytics import YOLO
import time, json, os, numpy as np

ATLANACAK = []   # Tamamlanan modelleri buraya ekle (kesinti sonrası devam için)

# ABLATION_ETIKET Hücre 4A'da tanımlanır. Tanımsızsa varsayılan ata:
if 'ABLATION_ETIKET' not in globals():
    ABLATION_ETIKET = 'v2'
    print(f'⚠️  ABLATION_ETIKET tanımsızdı → "{ABLATION_ETIKET}" atandı')

os.makedirs(MODEL_DIZIN, exist_ok=True)

# Önceki sonuçları yükle (varsa)
if os.path.exists(SONUC_JSON):
    with open(SONUC_JSON) as f:
        sonuclar = json.load(f)
    print(f'📂 Önceki sonuçlar yüklendi: {len(sonuclar)} model')
else:
    sonuclar = {}

print('=' * 70)
print(f'🏆 KARŞILAŞTIRMALI EĞİTİM — {len(KARSILASTIRMA_MODELLERI)} MODEL')
print('=' * 70)
print(f'   Çözünürlük : {IMG_SIZE}x{IMG_SIZE}')
print(f'   Epoch      : {EPOCHS}')
print(f'   Ablation   : {ABLATION_ETIKET}')
print(f'   Veri       : {HASAR_DATA_YAML}')
print('=' * 70)

genel_baslangic = time.time()

for model_id, model_adi, batch in KARSILASTIRMA_MODELLERI:
    # Ablation etiketiyle birlikte anahtar oluştur
    sonuc_key = f'{model_id}__{ABLATION_ETIKET}'

    if model_id in ATLANACAK or sonuc_key in sonuclar:
        print(f'\n⏭️  {model_adi} atlandı (zaten var)')
        continue

    print(f'\n{"─" * 70}')
    print(f'🚀 {model_adi} ({model_id}) — batch={batch}, imgsz={IMG_SIZE}')
    print(f'{"─" * 70}')

    t0 = time.time()

    try:
        model = YOLO(f'{model_id}.pt')
        model.train(
            data=HASAR_DATA_YAML,
            epochs=EPOCHS,                  # 100
            imgsz=IMG_SIZE,                 # 960
            batch=batch,
            device=DEVICE,
            project=MODEL_DIZIN,
            name=f'hasar_{model_id}_{ABLATION_ETIKET}',
            exist_ok=True,
            pretrained=True,
            optimizer='AdamW',
            lr0=0.001,
            lrf=0.01,
            patience=PATIENCE,              # 30
            workers=4,
            cache=False,
            # ─── AUGMENTATION (GÜNCELLENDİ) ────────────────────────────────
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
            fliplr=0.5, flipud=0.0,
            degrees=5.0,                    # YENİ: hafif rotasyon
            mosaic=0.5,                     # 0.0'dan AÇILDI
            close_mosaic=15,                # son 15 epoch mozaik kapanır
            copy_paste=0.3,                 # YENİ: segmentasyonda çok etkili
            scale=0.4, translate=0.1,
            erasing=0.2,                    # YENİ: random erasing
            # ─── KAYIP AĞIRLIKLARI ─────────────────────────────────────────
            box=7.5,
            cls=0.6,                        # 0.5'ten artırıldı (sınıf ayrımı)
            dfl=1.5,
            plots=True,
            verbose=True,
        )
        egitim_suresi = time.time() - t0

        # ─── TEST SETİNDE DEĞERLENDİR ─────────────────────────────────────────
        best_path = os.path.join(MODEL_DIZIN, f'hasar_{model_id}_{ABLATION_ETIKET}',
                                 'weights', 'best.pt')
        best = YOLO(best_path)
        metrics = best.val(data=HASAR_DATA_YAML, split='test',
                           imgsz=IMG_SIZE, verbose=False)

        # ─── INFERENCE HIZI ÖLÇ ───────────────────────────────────────────────
        dummy = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        best.predict(dummy, imgsz=IMG_SIZE, verbose=False)   # warmup
        t1 = time.time()
        for _ in range(20):
            best.predict(dummy, imgsz=IMG_SIZE, verbose=False)
        inference_ms = (time.time() - t1) / 20 * 1000

        model_mb = os.path.getsize(best_path) / 1e6

        sonuclar[sonuc_key] = {
            'ad':            f'{model_adi} ({ABLATION_ETIKET})',
            'model_id':      model_id,
            'ablation':      ABLATION_ETIKET,
            'imgsz':         IMG_SIZE,
            'epochs':        EPOCHS,
            'box_mAP50':     float(metrics.box.map50),
            'box_mAP50_95':  float(metrics.box.map),
            'box_precision': float(metrics.box.mp),
            'box_recall':    float(metrics.box.mr),
            'mask_mAP50':    float(metrics.seg.map50),
            'mask_mAP50_95': float(metrics.seg.map),
            'egitim_dk':     egitim_suresi / 60,
            'inference_ms':  inference_ms,
            'model_mb':      model_mb,
            'path':          best_path,
        }

        # Her modelden sonra kaydet
        with open(SONUC_JSON, 'w') as f:
            json.dump(sonuclar, f, indent=2)

        r = sonuclar[sonuc_key]
        print(f'\n✅ {model_adi} TAMAMLANDI')
        print(f'   Box  mAP50    : {r["box_mAP50"]:.4f}')
        print(f'   Box  mAP50-95 : {r["box_mAP50_95"]:.4f}')
        print(f'   Mask mAP50    : {r["mask_mAP50"]:.4f}')
        print(f'   Mask mAP50-95 : {r["mask_mAP50_95"]:.4f}')
        print(f'   Precision     : {r["box_precision"]:.4f}')
        print(f'   Recall        : {r["box_recall"]:.4f}')
        print(f'   Inference     : {r["inference_ms"]:.1f} ms')
        print(f'   Süre          : {r["egitim_dk"]:.1f} dk')

        # Sınıf bazlı hızlı özet
        print(f'\n   📋 SINIF BAZLI (Mask mAP50):')
        for i, sinif in enumerate(HASAR_SINIFLAR):
            try:
                _, _, sap50, _ = metrics.seg.class_result(i)
                durum = '✅' if sap50 >= 0.5 else ('⚠️ ' if sap50 >= 0.35 else '❌')
                print(f'      {durum} {sinif:8s}: {sap50:.4f}')
            except Exception:
                pass

    except Exception as e:
        print(f'\n❌ {model_adi} HATA: {str(e)[:250]}')
        print('   💡 CUDA out of memory ise batch değerini düşür (Hücre 1).')
        continue

toplam_dk = (time.time() - genel_baslangic) / 60
print('\n' + '=' * 70)
print(f'🎉 KARŞILAŞTIRMA TAMAMLANDI — {len(sonuclar)} model, {toplam_dk:.0f} dk')
print(f'   Sonuçlar: {SONUC_JSON}')
print('=' * 70)
print('\n💡 SONRAKİ ADIM: Hücre 6 (karşılaştırma) → Hücre 6B (confusion matrix)')

🏆 KARŞILAŞTIRMALI EĞİTİM — 5 MODEL

──────────────────────────────────────────────────────────────────────
🚀 YOLOv11-Nano (yolo11n-seg) — batch=16
──────────────────────────────────────────────────────────────────────
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/My-First-Project-2-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, h

---
## 📊 HÜCRE 6: Karşılaştırma Tablosu + Grafikler (Tez İçin)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📊 HÜCRE 6: KARŞILAŞTIRMA ANALİZİ
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 ÇIKTILAR (tez "Bulgular" bölümü için):
#    - Karşılaştırma tablosu (Excel + LaTeX)
#    - 4 panelli grafik (mAP, hız, boyut, doğruluk-hız dengesi)
#    - En iyi model seçimi
# ═══════════════════════════════════════════════════════════════════════════════

import json, pandas as pd, matplotlib.pyplot as plt, numpy as np

with open(SONUC_JSON) as f:
    sonuclar = json.load(f)

if not sonuclar:
    raise RuntimeError('Sonuç yok! Önce Hücre 5\'i çalıştır.')

# ─── TABLO ────────────────────────────────────────────────────────────────────
df = pd.DataFrame([
    {
        'Model':        v['ad'],
        'Box mAP50':    round(v['box_mAP50'], 4),
        'Box mAP50-95': round(v['box_mAP50_95'], 4),
        'Mask mAP50':   round(v['mask_mAP50'], 4),
        'Mask mAP50-95':round(v['mask_mAP50_95'], 4),
        'Precision':    round(v['box_precision'], 4),
        'Recall':       round(v['box_recall'], 4),
        'Inference(ms)':round(v['inference_ms'], 1),
        'Boyut(MB)':    round(v['model_mb'], 1),
        'Eğitim(dk)':   round(v['egitim_dk'], 1),
    }
    for v in sonuclar.values()
]).sort_values('Mask mAP50', ascending=False).reset_index(drop=True)

print('=' * 100)
print('📊 MODEL KARŞILAŞTIRMA TABLOSU')
print('=' * 100)
print(df.to_string(index=False))
print('=' * 100)

# Excel + LaTeX kaydet
df.to_excel(f'{OUTPUT_DIR}/model_karsilastirma.xlsx', index=False)
with open(f'{OUTPUT_DIR}/model_karsilastirma_latex.txt', 'w') as f:
    f.write(df.to_latex(index=False, float_format='%.4f'))
print(f'\n💾 Excel : {OUTPUT_DIR}/model_karsilastirma.xlsx')
print(f'💾 LaTeX : {OUTPUT_DIR}/model_karsilastirma_latex.txt')

# ─── EN İYİ MODEL ─────────────────────────────────────────────────────────────
en_iyi_id = max(sonuclar, key=lambda k: sonuclar[k]['mask_mAP50'])
EN_IYI = sonuclar[en_iyi_id]
HASAR_MODEL_PATH = EN_IYI['path']

print(f'\n🥇 EN İYİ MODEL: {EN_IYI["ad"]}')
print(f'   Mask mAP50 : {EN_IYI["mask_mAP50"]:.4f}')
print(f'   Inference  : {EN_IYI["inference_ms"]:.1f} ms')
print(f'   Path       : {HASAR_MODEL_PATH}')

# ─── GRAFİKLER ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Model Karşılaştırma Analizi — Hasar Tespiti',
             fontsize=16, fontweight='bold')

adlar = df['Model'].tolist()
x = np.arange(len(adlar))
renk = plt.cm.viridis(np.linspace(0.15, 0.85, len(adlar)))

# 1. mAP karşılaştırma
ax = axes[0, 0]
w = 0.35
ax.bar(x - w/2, df['Box mAP50'],  w, label='Box mAP50',  color='#3b82f6')
ax.bar(x + w/2, df['Mask mAP50'], w, label='Mask mAP50', color='#f59e0b')
ax.set_xticks(x); ax.set_xticklabels(adlar, rotation=20, ha='right')
ax.set_ylabel('mAP50'); ax.set_title('Doğruluk Karşılaştırması')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# 2. Inference hızı
ax = axes[0, 1]
bars = ax.bar(x, df['Inference(ms)'], color=renk)
ax.set_xticks(x); ax.set_xticklabels(adlar, rotation=20, ha='right')
ax.set_ylabel('ms'); ax.set_title('Inference Süresi (düşük = iyi)')
ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, df['Inference(ms)']):
    ax.text(b.get_x()+b.get_width()/2, v, f'{v:.0f}', ha='center', va='bottom', fontsize=9)

# 3. Model boyutu
ax = axes[1, 0]
bars = ax.bar(x, df['Boyut(MB)'], color=renk)
ax.set_xticks(x); ax.set_xticklabels(adlar, rotation=20, ha='right')
ax.set_ylabel('MB'); ax.set_title('Model Boyutu')
ax.grid(axis='y', alpha=0.3)

# 4. Doğruluk-Hız dengesi (scatter)
ax = axes[1, 1]
for i, r in df.iterrows():
    ax.scatter(r['Inference(ms)'], r['Mask mAP50'],
               s=r['Boyut(MB)']*6, alpha=0.65, color=renk[i],
               edgecolors='black', linewidth=1.2)
    ax.annotate(r['Model'], (r['Inference(ms)'], r['Mask mAP50']),
                xytext=(6, 6), textcoords='offset points', fontsize=9)
ax.set_xlabel('Inference (ms)'); ax.set_ylabel('Mask mAP50')
ax.set_title('Doğruluk-Hız Dengesi (daire = model boyutu)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_karsilastirma.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'\n💾 Grafik: {OUTPUT_DIR}/model_karsilastirma.png')

---
## 🔬 HÜCRE 6B: Detaylı Metrik Analizi (Tez İçin Kritik)

Her modelin **sınıf bazlı** performansı, confusion matrix, PR/F1 eğrileri ve eğitim grafikleri.

> 📌 **Not:** MSE/RMSE/MAE regresyon metrikleridir, nesne tespitinde kullanılmaz.
> Buradaki karşılıkları: **mAP50-95** (genel başarı), **Precision/Recall/F1** (doğruluk), **Confusion Matrix** (sınıf karışımı).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔬 HÜCRE 6B: DETAYLI METRİK ANALİZİ
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 ÜRETİLEN ÇIKTILAR (hepsi tez için kullanılabilir):
#    1. Sınıf bazlı metrik tablosu (her sınıf için P, R, F1, mAP50, mAP50-95)
#    2. Confusion Matrix (ham + normalize)
#    3. Precision-Recall eğrisi
#    4. F1-Confidence eğrisi
#    5. Eğitim eğrileri (loss + mAP epoch bazlı)
#    6. Örnek tahmin görselleri (ground truth vs prediction)
#
# 🎛️ AYAR:
#    ANALIZ_MODEL : Hangi model analiz edilecek. None = en iyi model
# ═══════════════════════════════════════════════════════════════════════════════

from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, os, json
from PIL import Image

ANALIZ_MODEL = None   # None = en iyi model, veya 'yolo11m-seg' gibi ver

if ANALIZ_MODEL is None:
    ANALIZ_MODEL = en_iyi_id

model_dir  = os.path.join(MODEL_DIZIN, f'hasar_{ANALIZ_MODEL}')
model_path = f'{model_dir}/weights/best.pt'
model_adi  = sonuclar[ANALIZ_MODEL]['ad']

print('=' * 70)
print(f'🔬 DETAYLI ANALİZ: {model_adi}')
print('=' * 70)

# ══ 1. SINIF BAZLI METRİKLER ═══════════════════════════════════════════════════
print('\n📊 Test setinde değerlendiriliyor...')
m = YOLO(model_path)
metrics = m.val(data=HASAR_DATA_YAML, split='test', verbose=False,
                plots=True, save_json=False,
                project=MODEL_DIZIN, name=f'val_{ANALIZ_MODEL}',
                exist_ok=True)

val_dir = os.path.join(MODEL_DIZIN, f'val_{ANALIZ_MODEL}')

# Sınıf bazlı tablo oluştur
satirlar = []
for i, sinif in enumerate(HASAR_SINIFLAR):
    try:
        # Box metrikleri
        bp, br, bap50, bap = metrics.box.class_result(i)
        # Mask metrikleri
        sp, sr, sap50, sap = metrics.seg.class_result(i)

        f1_box  = 2 * bp * br / (bp + br) if (bp + br) > 0 else 0
        f1_mask = 2 * sp * sr / (sp + sr) if (sp + sr) > 0 else 0

        satirlar.append({
            'Sınıf':          sinif,
            'Box Precision':  round(bp, 4),
            'Box Recall':     round(br, 4),
            'Box F1':         round(f1_box, 4),
            'Box mAP50':      round(bap50, 4),
            'Box mAP50-95':   round(bap, 4),
            'Mask Precision': round(sp, 4),
            'Mask Recall':    round(sr, 4),
            'Mask F1':        round(f1_mask, 4),
            'Mask mAP50':     round(sap50, 4),
            'Mask mAP50-95':  round(sap, 4),
        })
    except Exception as e:
        print(f'   ⚠️ {sinif} metrikleri alınamadı: {str(e)[:60]}')

# Ortalama satırı
if satirlar:
    df_sinif = pd.DataFrame(satirlar)
    ort = {'Sınıf': 'ORTALAMA'}
    for c in df_sinif.columns[1:]:
        ort[c] = round(df_sinif[c].mean(), 4)
    df_sinif = pd.concat([df_sinif, pd.DataFrame([ort])], ignore_index=True)

    print('\n' + '=' * 100)
    print('📋 SINIF BAZLI METRİK TABLOSU')
    print('=' * 100)
    print(df_sinif.to_string(index=False))
    print('=' * 100)

    df_sinif.to_excel(f'{OUTPUT_DIR}/sinif_bazli_metrikler.xlsx', index=False)
    with open(f'{OUTPUT_DIR}/sinif_bazli_metrikler_latex.txt', 'w') as f:
        f.write(df_sinif.to_latex(index=False, float_format='%.4f'))
    print(f'\n💾 Excel : {OUTPUT_DIR}/sinif_bazli_metrikler.xlsx')
    print(f'💾 LaTeX : {OUTPUT_DIR}/sinif_bazli_metrikler_latex.txt')

# ══ 2-5. YOLO'NUN ÜRETTİĞİ GRAFİKLERİ GÖSTER ══════════════════════════════════
print('\n' + '=' * 70)
print('📈 GRAFİKLER')
print('=' * 70)

grafik_listesi = [
    ('confusion_matrix.png',            'Confusion Matrix (Ham Sayılar)'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix (Normalize)'),
    ('BoxPR_curve.png',                 'Precision-Recall Eğrisi (Box)'),
    ('MaskPR_curve.png',                'Precision-Recall Eğrisi (Mask)'),
    ('BoxF1_curve.png',                 'F1-Confidence Eğrisi (Box)'),
    ('MaskF1_curve.png',                'F1-Confidence Eğrisi (Mask)'),
    ('BoxP_curve.png',                  'Precision-Confidence Eğrisi'),
    ('BoxR_curve.png',                  'Recall-Confidence Eğrisi'),
]

bulunan = 0
for dosya, baslik in grafik_listesi:
    yol = os.path.join(val_dir, dosya)
    if os.path.exists(yol):
        plt.figure(figsize=(11, 8))
        plt.imshow(Image.open(yol))
        plt.title(f'{baslik} — {model_adi}', fontsize=13, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/{dosya}', dpi=180, bbox_inches='tight')
        plt.show()
        bulunan += 1

print(f'\n✅ {bulunan} grafik gösterildi ve {OUTPUT_DIR} klasörüne kaydedildi.')

# ══ 6. EĞİTİM EĞRİLERİ ═════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('📉 EĞİTİM EĞRİLERİ')
print('=' * 70)

results_png = os.path.join(model_dir, 'results.png')
if os.path.exists(results_png):
    plt.figure(figsize=(16, 9))
    plt.imshow(Image.open(results_png))
    plt.title(f'Eğitim Süreci — {model_adi}', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/egitim_egrileri_{ANALIZ_MODEL}.png', dpi=180, bbox_inches='tight')
    plt.show()

# results.csv'den özel grafik çiz (daha temiz görünüm için)
results_csv = os.path.join(model_dir, 'results.csv')
if os.path.exists(results_csv):
    dfr = pd.read_csv(results_csv)
    dfr.columns = dfr.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Eğitim Metrikleri — {model_adi}', fontsize=14, fontweight='bold')

    # Loss eğrileri
    ax = axes[0]
    for col, lbl in [('train/box_loss', 'Box Loss'),
                     ('train/seg_loss', 'Seg Loss'),
                     ('train/cls_loss', 'Cls Loss')]:
        if col in dfr.columns:
            ax.plot(dfr['epoch'], dfr[col], label=lbl, linewidth=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('Eğitim Kayıpları'); ax.legend(); ax.grid(alpha=0.3)

    # mAP eğrileri
    ax = axes[1]
    for col, lbl in [('metrics/mAP50(B)', 'Box mAP50'),
                     ('metrics/mAP50-95(B)', 'Box mAP50-95'),
                     ('metrics/mAP50(M)', 'Mask mAP50'),
                     ('metrics/mAP50-95(M)', 'Mask mAP50-95')]:
        if col in dfr.columns:
            ax.plot(dfr['epoch'], dfr[col], label=lbl, linewidth=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
    ax.set_title('Doğrulama mAP'); ax.legend(); ax.grid(alpha=0.3)

    # Precision-Recall
    ax = axes[2]
    for col, lbl in [('metrics/precision(B)', 'Precision'),
                     ('metrics/recall(B)', 'Recall')]:
        if col in dfr.columns:
            ax.plot(dfr['epoch'], dfr[col], label=lbl, linewidth=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Değer')
    ax.set_title('Precision & Recall'); ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/egitim_metrikleri_detay.png', dpi=180, bbox_inches='tight')
    plt.show()

    # Son epoch değerleri
    son = dfr.iloc[-1]
    print(f'\n📌 SON EPOCH DEĞERLERİ (epoch {int(son["epoch"])}):')
    for col in dfr.columns:
        if col.startswith('metrics/'):
            print(f'   {col.replace("metrics/", ""):22s}: {son[col]:.4f}')

# ══ 7. ÖRNEK TAHMİNLER (Ground Truth vs Prediction) ═══════════════════════════
print('\n' + '=' * 70)
print('🖼️  ÖRNEK TAHMİNLER')
print('=' * 70)

for dosya, baslik in [('val_batch0_labels.jpg', 'Gerçek Etiketler (Ground Truth)'),
                      ('val_batch0_pred.jpg',   'Model Tahminleri (Prediction)')]:
    yol = os.path.join(val_dir, dosya)
    if not os.path.exists(yol):
        yol = os.path.join(model_dir, dosya)
    if os.path.exists(yol):
        plt.figure(figsize=(15, 11))
        plt.imshow(Image.open(yol))
        plt.title(baslik, fontsize=13, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/{dosya.replace(".jpg", ".png")}', dpi=150, bbox_inches='tight')
        plt.show()

print('\n' + '=' * 70)
print('✅ DETAYLI ANALİZ TAMAMLANDI')
print('=' * 70)
print(f'   Tüm çıktılar: {OUTPUT_DIR}')
print('\n💡 TEZ İÇİN KULLANIM:')
print('   - sinif_bazli_metrikler.xlsx    → Bulgular tablosu')
print('   - confusion_matrix*.png          → Sınıf karışım analizi')
print('   - *PR_curve.png / *F1_curve.png  → Model davranışı yorumu')
print('   - egitim_metrikleri_detay.png    → Öğrenme süreci analizi')
print('   - val_batch0_*.png               → Nitel (kalitatif) değerlendirme')

---
## 🔁 HÜCRE 6C: K-Fold Çapraz Doğrulama (Kazanan Model İçin)

> 📌 **Ne zaman çalıştırılır?** Hücre 5 + 6 bittikten sonra, en iyi model belli olunca.
>
> **Neden?** Tek bölünme şansa bağlı olabilir. K-fold, modelin **kararlılığını** ölçer ve
> `mAP50 = 0.72 ± 0.03` gibi standart sapmalı sonuç verir — tez için çok daha güçlü.
>
> ⏱️ **Süre:** 5 fold × ~35 dk = ~3 saat (A100)

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔁 HÜCRE 6C: K-FOLD ÇAPRAZ DOĞRULAMA
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NE YAPAR?
#    - Train+Valid verisini K parçaya böler (test seti dokunulmaz kalır)
#    - Her turda 1 parça validation, K-1 parça training olur
#    - K kez eğitim yapar, ortalama ± standart sapma verir
#
# 📐 ÖRNEK (K=5):
#    Fold 1: [VAL][   ][   ][   ][   ]
#    Fold 2: [   ][VAL][   ][   ][   ]
#    Fold 3: [   ][   ][VAL][   ][   ]  → her görsel 1 kez validation olur
#    Fold 4: [   ][   ][   ][VAL][   ]
#    Fold 5: [   ][   ][   ][   ][VAL]
#
# 💡 TEZ İÇİN DEĞERİ:
#    "mAP50 = 0.5841" yerine "mAP50 = 0.5841 ± 0.0187" yazabilirsin.
#    Düşük std = model kararlı, yüksek std = veri bölünmesine duyarlı.
#
# ⚠️ ÖNEMLİ NOTLAR:
#    - Hücre 6 çalıştırılmamışsa bile çalışır (modeli JSON'dan bulur)
#    - Augmentation ayarları HÜCRE 5 İLE AYNI (karşılaştırılabilirlik için şart)
#    - Kesinti olursa kaldığı folddan devam eder
#
# 🎛️ AYARLAR:
#    K             : Kaç fold (5 standart, 3 hızlı, 10 en güçlü)
#    KFOLD_MODEL   : Hangi model. None = en iyi model otomatik
#    KFOLD_EPOCHS  : Fold başına epoch
# ═══════════════════════════════════════════════════════════════════════════════

import os, shutil, yaml, json, time, random
import numpy as np, pandas as pd
from pathlib import Path
from ultralytics import YOLO

K            = 5
KFOLD_MODEL  = None    # None = otomatik seç (en iyi model)
KFOLD_EPOCHS = 60      # Fold başına epoch

# ══ MODEL SEÇİMİ (Hücre 6'ya bağımlı değil) ════════════════════════════════════
if KFOLD_MODEL is None:
    # 1. Öncelik: Hücre 6 çalıştıysa en_iyi_id kullan
    if 'en_iyi_id' in globals():
        secilen_key = en_iyi_id
        print(f'✅ Hücre 6\'dan alındı: {secilen_key}')
    # 2. Yedek: JSON'dan en iyisini bul
    elif os.path.exists(SONUC_JSON):
        with open(SONUC_JSON) as f:
            _s = json.load(f)
        if not _s:
            raise RuntimeError('❌ JSON boş! Önce Hücre 5\'i çalıştır.')
        secilen_key = max(_s, key=lambda k: _s[k]['mask_mAP50'])
        print(f'✅ JSON\'dan seçildi: {secilen_key} '
              f'(mAP50={_s[secilen_key]["mask_mAP50"]:.4f})')
    else:
        raise RuntimeError('❌ Sonuç JSON\'u yok! Önce Hücre 5\'i çalıştır.')

    # Anahtar "yolo11m-seg__kendi_veri_plus_harici" formatında olabilir
    # → gerçek model id'sini ayıkla
    if '__' in secilen_key:
        KFOLD_MODEL = secilen_key.split('__')[0]
    else:
        KFOLD_MODEL = secilen_key
else:
    print(f'✅ Elle belirlendi: {KFOLD_MODEL}')

# ABLATION_ETIKET kontrolü
if 'ABLATION_ETIKET' not in globals():
    ABLATION_ETIKET = 'v2'

# Batch değerini bul (960 çözünürlük için düşük olmalı)
_batch_map = {m[0]: m[2] for m in KARSILASTIRMA_MODELLERI}
KFOLD_BATCH = _batch_map.get(KFOLD_MODEL, 8)

print('=' * 70)
print(f'🔁 {K}-FOLD ÇAPRAZ DOĞRULAMA')
print('=' * 70)
print(f'   Model       : {KFOLD_MODEL}')
print(f'   Batch       : {KFOLD_BATCH}')
print(f'   Çözünürlük  : {IMG_SIZE}')
print(f'   Epoch/fold  : {KFOLD_EPOCHS}')
print(f'   Ablation    : {ABLATION_ETIKET}')
print(f'   Tahmini süre: ~{K * KFOLD_EPOCHS * 1.5:.0f} dk')
print('=' * 70)

# ══ 1. TÜM TRAIN+VALID GÖRSELLERİNİ TOPLA ══════════════════════════════════════
havuz = []
for split in ['train', 'valid']:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    if not os.path.exists(img_dir):
        continue
    for ip in Path(img_dir).glob('*.*'):
        lp = os.path.join(lbl_dir, ip.stem + '.txt')
        if os.path.exists(lp):
            havuz.append((str(ip), lp))

if not havuz:
    raise RuntimeError('❌ Havuz boş! DATASET_DIR doğru mu kontrol et.')

random.seed(42)
random.shuffle(havuz)

test_sayisi = len(list(Path(os.path.join(DATASET_DIR, "test", "images")).glob("*.*")))
print(f'\n📦 Havuz: {len(havuz)} görsel (train + valid)')
print(f'   Test seti ({test_sayisi} görsel) dokunulmadı — final değerlendirme için ayrı.')

# ══ 2. K PARÇAYA BÖL ═══════════════════════════════════════════════════════════
fold_boyut = len(havuz) // K
foldlar = [havuz[i*fold_boyut : (i+1)*fold_boyut] for i in range(K)]
# Artanları son folda ekle
if len(havuz) % K:
    foldlar[-1].extend(havuz[K*fold_boyut:])

print(f'📐 Fold boyutları: {[len(f) for f in foldlar]}')

# ══ 3. HER FOLD İÇİN EĞİT ══════════════════════════════════════════════════════
KFOLD_DIR  = '/content/kfold_calisma'
KFOLD_JSON = os.path.join(ANA_DIZIN, f'kfold_{KFOLD_MODEL}_{ABLATION_ETIKET}.json')

kfold_sonuc = {}
if os.path.exists(KFOLD_JSON):
    with open(KFOLD_JSON) as f:
        kfold_sonuc = json.load(f)
    print(f'\n📂 Önceki fold sonuçları yüklendi: {len(kfold_sonuc)}/{K}')

genel_t0 = time.time()

for fold_no in range(K):
    if str(fold_no) in kfold_sonuc:
        print(f'\n⏭️  Fold {fold_no+1} atlandı (zaten var)')
        continue

    print(f'\n{"─" * 70}')
    print(f'🔁 FOLD {fold_no+1}/{K}')
    print(f'{"─" * 70}')

    # Klasör yapısını kur
    shutil.rmtree(KFOLD_DIR, ignore_errors=True)
    for s in ['train', 'valid']:
        os.makedirs(f'{KFOLD_DIR}/{s}/images', exist_ok=True)
        os.makedirs(f'{KFOLD_DIR}/{s}/labels', exist_ok=True)

    # Dosyaları dağıt (symlink → disk tasarrufu, desteklenmezse kopyala)
    for i, fold in enumerate(foldlar):
        hedef = 'valid' if i == fold_no else 'train'
        for ip, lp in fold:
            in_ = f'{KFOLD_DIR}/{hedef}/images/{os.path.basename(ip)}'
            ln_ = f'{KFOLD_DIR}/{hedef}/labels/{os.path.basename(lp)}'
            try:
                if not os.path.exists(in_):
                    os.symlink(ip, in_)
                if not os.path.exists(ln_):
                    os.symlink(lp, ln_)
            except OSError:
                # Symlink desteklenmiyorsa (bazı Drive mount'larında) kopyala
                if not os.path.exists(in_):
                    shutil.copy(ip, in_)
                if not os.path.exists(ln_):
                    shutil.copy(lp, ln_)

    n_tr = len(os.listdir(f'{KFOLD_DIR}/train/images'))
    n_va = len(os.listdir(f'{KFOLD_DIR}/valid/images'))
    print(f'   Train: {n_tr} | Valid: {n_va}')

    # data.yaml
    fold_yaml = f'{KFOLD_DIR}/data.yaml'
    with open(fold_yaml, 'w') as f:
        yaml.dump({
            'train': f'{KFOLD_DIR}/train/images',
            'val':   f'{KFOLD_DIR}/valid/images',
            'nc':    len(HASAR_SINIFLAR),
            'names': HASAR_SINIFLAR,
        }, f, default_flow_style=False, sort_keys=False)

    # ─── EĞİT ─────────────────────────────────────────────────────────────────
    batch = KFOLD_BATCH
    t0 = time.time()

    mdl = YOLO(f'{KFOLD_MODEL}.pt')
    mdl.train(
        data=fold_yaml,
        epochs=KFOLD_EPOCHS,
        imgsz=IMG_SIZE,
        batch=batch,
        device=DEVICE,
        project=MODEL_DIZIN,
        name=f'kfold_{KFOLD_MODEL}_f{fold_no+1}',
        exist_ok=True,
        pretrained=True,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        patience=20,
        workers=4,
        cache=False,
        # ─── HÜCRE 5 İLE AYNI AUGMENTATION (tutarlılık şart!) ──────────────
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
        fliplr=0.5, flipud=0.0,
        degrees=5.0,
        mosaic=0.5,
        close_mosaic=15,
        copy_paste=0.3,
        scale=0.4, translate=0.1,
        erasing=0.2,
        box=7.5, cls=0.6, dfl=1.5,
        plots=False,
        verbose=False,
    )

    # ─── BU FOLD'UN VALIDATION SETİNDE DEĞERLENDİR ────────────────────────────
    best = YOLO(os.path.join(MODEL_DIZIN, f'kfold_{KFOLD_MODEL}_f{fold_no+1}',
                             'weights', 'best.pt'))
    mt = best.val(data=fold_yaml, split='val', imgsz=IMG_SIZE, verbose=False)

    # Sınıf bazlı sonuçları da kaydet
    sinif_sonuc = {}
    for ci, sinif in enumerate(HASAR_SINIFLAR):
        try:
            _, _, sap50, _ = mt.seg.class_result(ci)
            sinif_sonuc[sinif] = float(sap50)
        except Exception:
            sinif_sonuc[sinif] = 0.0

    kfold_sonuc[str(fold_no)] = {
        'fold':          fold_no + 1,
        'box_mAP50':     float(mt.box.map50),
        'box_mAP50_95':  float(mt.box.map),
        'mask_mAP50':    float(mt.seg.map50),
        'mask_mAP50_95': float(mt.seg.map),
        'precision':     float(mt.box.mp),
        'recall':        float(mt.box.mr),
        'sinif_mAP50':   sinif_sonuc,
        'sure_dk':       (time.time() - t0) / 60,
        'n_train':       n_tr,
        'n_valid':       n_va,
    }

    with open(KFOLD_JSON, 'w') as f:
        json.dump(kfold_sonuc, f, indent=2)

    r = kfold_sonuc[str(fold_no)]
    print(f'   ✅ Box mAP50: {r["box_mAP50"]:.4f} | '
          f'Mask mAP50: {r["mask_mAP50"]:.4f} | {r["sure_dk"]:.0f} dk')
    print(f'      Sınıf bazlı: ' +
          ' | '.join(f'{k}={v:.3f}' for k, v in sinif_sonuc.items()))

# ══ 4. SONUÇ ANALİZİ ═══════════════════════════════════════════════════════════
df_kf = pd.DataFrame([
    {k: v for k, v in d.items() if k != 'sinif_mAP50'}
    for d in kfold_sonuc.values()
]).sort_values('fold')

print('\n' + '=' * 90)
print(f'📊 {K}-FOLD SONUÇLARI — {KFOLD_MODEL}')
print('=' * 90)
goster = df_kf[['fold', 'box_mAP50', 'box_mAP50_95', 'mask_mAP50',
                'mask_mAP50_95', 'precision', 'recall']].round(4)
print(goster.to_string(index=False))
print('─' * 90)

# Ortalama ± std
metrikler = ['box_mAP50', 'box_mAP50_95', 'mask_mAP50', 'mask_mAP50_95',
             'precision', 'recall']
print(f'{"Metrik":18s} {"Ortalama":>10s} {"Std Sapma":>11s} {"Min":>8s} {"Max":>8s}')
print('─' * 90)
ozet_satirlar = []
for m in metrikler:
    ort, std = df_kf[m].mean(), df_kf[m].std()
    print(f'{m:18s} {ort:10.4f} {std:11.4f} {df_kf[m].min():8.4f} {df_kf[m].max():8.4f}')
    ozet_satirlar.append({
        'Metrik': m, 'Ortalama': round(ort, 4), 'Std': round(std, 4),
        'Min': round(df_kf[m].min(), 4), 'Max': round(df_kf[m].max(), 4),
        'Tez Formatı': f'{ort:.4f} ± {std:.4f}',
    })
print('=' * 90)

df_ozet = pd.DataFrame(ozet_satirlar)

# ─── SINIF BAZLI ORTALAMA ─────────────────────────────────────────────────────
print(f'\n🏷️  SINIF BAZLI ORTALAMA (Mask mAP50):')
sinif_ozet = []
for sinif in HASAR_SINIFLAR:
    degerler = [d['sinif_mAP50'][sinif] for d in kfold_sonuc.values()
                if 'sinif_mAP50' in d]
    if degerler:
        ort, std = np.mean(degerler), np.std(degerler, ddof=1) if len(degerler) > 1 else 0
        durum = '✅' if ort >= 0.5 else ('⚠️ ' if ort >= 0.35 else '❌')
        print(f'   {durum} {sinif:8s}: {ort:.4f} ± {std:.4f}')
        sinif_ozet.append({
            'Sınıf': sinif, 'Ortalama': round(ort, 4), 'Std': round(std, 4),
            'Tez Formatı': f'{ort:.4f} ± {std:.4f}',
        })
df_sinif_kf = pd.DataFrame(sinif_ozet)

# ─── KARARLILIK YORUMU ────────────────────────────────────────────────────────
cv = df_kf['mask_mAP50'].std() / df_kf['mask_mAP50'].mean() * 100
print(f'\n📌 Varyasyon Katsayısı (Mask mAP50): %{cv:.2f}')
if cv < 5:
    print('   ✅ Model ÇOK KARARLI — veri bölünmesinden neredeyse etkilenmiyor.')
elif cv < 10:
    print('   ✅ Model KARARLI — kabul edilebilir varyasyon.')
else:
    print('   ⚠️  Model veri bölünmesine DUYARLI — daha fazla veri faydalı olabilir.')

# ══ 5. GRAFİK ══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
fig.suptitle(f'{K}-Fold Çapraz Doğrulama — {KFOLD_MODEL}',
             fontsize=14, fontweight='bold')

# 1. Fold bazlı çizgi
ax = axes[0]
for m, lbl, c in [('box_mAP50', 'Box mAP50', '#3b82f6'),
                  ('mask_mAP50', 'Mask mAP50', '#f59e0b')]:
    ax.plot(df_kf['fold'], df_kf[m], 'o-', label=lbl, linewidth=2,
            markersize=8, color=c)
    ax.axhline(df_kf[m].mean(), linestyle='--', alpha=0.5, color=c)
ax.set_xlabel('Fold'); ax.set_ylabel('mAP50')
ax.set_xticks(df_kf['fold'])
ax.set_title('Fold Bazlı Performans'); ax.legend(); ax.grid(alpha=0.3)

# 2. Box plot
ax = axes[1]
bp = ax.boxplot([df_kf['box_mAP50'], df_kf['mask_mAP50'],
                 df_kf['precision'], df_kf['recall']],
                labels=['Box\nmAP50', 'Mask\nmAP50', 'Precision', 'Recall'],
                patch_artist=True)
for patch, c in zip(bp['boxes'], ['#3b82f6', '#f59e0b', '#10b981', '#8b5cf6']):
    patch.set_facecolor(c); patch.set_alpha(0.65)
ax.set_ylabel('Değer'); ax.set_title('Metrik Dağılımları'); ax.grid(axis='y', alpha=0.3)

# 3. Sınıf bazlı
ax = axes[2]
if sinif_ozet:
    adlar = [s['Sınıf'] for s in sinif_ozet]
    ortl  = [s['Ortalama'] for s in sinif_ozet]
    stdl  = [s['Std'] for s in sinif_ozet]
    ax.bar(adlar, ortl, yerr=stdl, capsize=6,
           color=['#ef4444', '#3b82f6', '#f59e0b'][:len(adlar)],
           alpha=0.8, edgecolor='black', linewidth=1)
    ax.axhline(0.5, linestyle='--', color='gray', alpha=0.7, label='Hedef (0.50)')
    ax.set_ylabel('Mask mAP50'); ax.set_title('Sınıf Bazlı Performans')
    ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/kfold_analizi.png', dpi=180, bbox_inches='tight')
plt.show()

# ══ 6. KAYDET ══════════════════════════════════════════════════════════════════
with pd.ExcelWriter(f'{OUTPUT_DIR}/kfold_sonuclari.xlsx', engine='openpyxl') as w:
    goster.to_excel(w, sheet_name='Fold Detay', index=False)
    df_ozet.to_excel(w, sheet_name='Özet (Ort±Std)', index=False)
    if not df_sinif_kf.empty:
        df_sinif_kf.to_excel(w, sheet_name='Sınıf Bazlı', index=False)

with open(f'{OUTPUT_DIR}/kfold_latex.txt', 'w') as f:
    f.write('% Genel metrikler\n')
    f.write(df_ozet.to_latex(index=False))
    if not df_sinif_kf.empty:
        f.write('\n\n% Sınıf bazlı\n')
        f.write(df_sinif_kf.to_latex(index=False))

print(f'\n💾 Excel : {OUTPUT_DIR}/kfold_sonuclari.xlsx')
print(f'💾 LaTeX : {OUTPUT_DIR}/kfold_latex.txt')
print(f'💾 Grafik: {OUTPUT_DIR}/kfold_analizi.png')

# ══ 7. TEZ İÇİN HAZIR CÜMLE ════════════════════════════════════════════════════
print(f'\n' + '=' * 70)
print('📝 TEZDE ŞÖYLE YAZABİLİRSİN:')
print('=' * 70)
mm, ms = df_kf['mask_mAP50'].mean(), df_kf['mask_mAP50'].std()
bm, bs = df_kf['box_mAP50'].mean(), df_kf['box_mAP50'].std()
print(f'   "Önerilen modelin genelleme başarısı {K}-katlı çapraz doğrulama')
print(f'    ile değerlendirilmiştir. Model, Box mAP50 = {bm:.4f} ± {bs:.4f}')
print(f'    ve Mask mAP50 = {mm:.4f} ± {ms:.4f} değerlerine ulaşmıştır.')
print(f'    Varyasyon katsayısının %{cv:.2f} olması, modelin veri bölünmesine')
print(f'    karşı kararlı davrandığını göstermektedir."')
print('=' * 70)

# ══ 8. TEMİZLİK ════════════════════════════════════════════════════════════════
shutil.rmtree(KFOLD_DIR, ignore_errors=True)
toplam_dk = (time.time() - genel_t0) / 60
print(f'\n🧹 Geçici klasör temizlendi.')
print(f'⏱️  Toplam süre: {toplam_dk:.0f} dakika')

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


NameError: name 'en_iyi_id' is not defined

---
# ⚡ HÜCRE 6D: DRIVE'DAN MODEL YÜKLE (Eğitimi Atla)

> 🎯 **Modelleri daha önce eğittiysen**, Hücre 2-6'yı çalıştırmana gerek yok.
>
> **Kısa yol:** `Hücre 0` → `Hücre 1` → `Hücre 6D` → `Hücre 7` → `Hücre 8/9/10`
>
> Bu hücre Drive'daki modelleri bulur, yükler ve gerekli tüm değişkenleri hazırlar.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ⚡ HÜCRE 6D: DRIVE'DAN HAZIR MODELLERİ YÜKLE
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NE YAPAR?
#    - Drive'daki tüm eğitilmiş modelleri tarar ve listeler
#    - Konum modelini bulur (KONUM_MODEL_PATH)
#    - Hasar modelini seçer (HASAR_MODEL_PATH)
#    - Karşılaştırma sonuçlarını yükler (EN_IYI)
#    - Hücre 7'nin ihtiyaç duyduğu HER ŞEYİ hazırlar
#
# ⚡ KISA YOL:
#    Hücre 0 → Hücre 1 → BU HÜCRE → Hücre 7 → Hücre 8/9/10
#    (Hücre 2, 3, 4, 5, 6 atlanır — eğitim yapılmaz)
#
# 🎛️ AYAR:
#    SECILEN_HASAR_MODEL:
#      None          → En iyi mAP50'ye sahip modeli otomatik seçer
#      'yolo11m-seg' → Belirli bir modeli zorla seçer
# ═══════════════════════════════════════════════════════════════════════════════

import os, glob, json
from ultralytics import YOLO

SECILEN_HASAR_MODEL = None   # None = otomatik en iyi

print('=' * 70)
print('⚡ DRIVE\'DAN MODEL YÜKLEME')
print('=' * 70)

if 'MODEL_DIZIN' not in globals():
    raise RuntimeError('❌ MODEL_DIZIN tanımsız! Önce HÜCRE 0\'ı çalıştır.')

print(f'📁 Taranan dizin: {MODEL_DIZIN}\n')

# ══ 1. TÜM MODELLERİ BUL ═══════════════════════════════════════════════════════
tum_modeller = sorted(glob.glob(f'{MODEL_DIZIN}/**/weights/best.pt', recursive=True))

if not tum_modeller:
    print('❌ HİÇ MODEL BULUNAMADI!')
    print(f'   {MODEL_DIZIN} klasörü boş.')
    print('\n   Yapman gerekenler:')
    print('   1. Hücre 2-3 → Konum modelini eğit')
    print('   2. Hücre 4-5 → Hasar modellerini eğit')
    raise RuntimeError('Model yok — önce eğitim yap.')

print(f'🔍 {len(tum_modeller)} model bulundu:\n')
konum_adaylari, hasar_adaylari = [], []

for mp in tum_modeller:
    klasor = mp.split('/')[-3]
    boyut  = os.path.getsize(mp) / 1e6
    if klasor.startswith('konum'):
        konum_adaylari.append(mp)
        tip = '🚙 KONUM'
    elif klasor.startswith('hasar'):
        hasar_adaylari.append(mp)
        tip = '💥 HASAR'
    else:
        tip = '❓ DİĞER'
    print(f'   {tip}  {klasor:30s} {boyut:6.1f} MB')

# ══ 2. KONUM MODELİNİ SEÇ ══════════════════════════════════════════════════════
print('\n' + '─' * 70)
if konum_adaylari:
    KONUM_MODEL_PATH = konum_adaylari[0]
    print(f'✅ KONUM MODELİ: {KONUM_MODEL_PATH.split("/")[-3]}')
else:
    print('⚠️  KONUM MODELİ YOK!')
    print('   Füzyon çalışmaz — parça bilgisi olmadan sadece hasar tespiti yapılır.')
    print('   Hücre 2-3\'ü çalıştırıp konum modelini eğitmen gerek.')
    KONUM_MODEL_PATH = None

# ══ 3. HASAR MODELİNİ SEÇ ══════════════════════════════════════════════════════
if not hasar_adaylari:
    raise RuntimeError('❌ Hasar modeli yok! Hücre 4-5\'i çalıştır.')

# Karşılaştırma sonuçları varsa yükle
sonuclar = {}
if os.path.exists(SONUC_JSON):
    with open(SONUC_JSON) as f:
        sonuclar = json.load(f)
    print(f'\n📊 Karşılaştırma sonuçları yüklendi ({len(sonuclar)} model):')
    for k, v in sorted(sonuclar.items(), key=lambda x: -x[1]['mask_mAP50']):
        print(f'   {v["ad"]:20s} mAP50={v["mask_mAP50"]:.4f}  '
              f'{v["inference_ms"]:5.1f}ms  {v["model_mb"]:5.1f}MB')

# Seçim yap
if SECILEN_HASAR_MODEL:
    # Kullanıcı belirli bir model istedi
    eslesen = [p for p in hasar_adaylari if SECILEN_HASAR_MODEL in p]
    if not eslesen:
        raise RuntimeError(f'❌ "{SECILEN_HASAR_MODEL}" bulunamadı.\n'
                           f'   Mevcut: {[p.split("/")[-3] for p in hasar_adaylari]}')
    HASAR_MODEL_PATH = eslesen[0]
    en_iyi_id = SECILEN_HASAR_MODEL
elif sonuclar:
    # JSON'dan en iyiyi seç
    en_iyi_id = max(sonuclar, key=lambda k: sonuclar[k]['mask_mAP50'])
    HASAR_MODEL_PATH = sonuclar[en_iyi_id].get('path', '')
    # Path Drive'a taşınmışsa düzelt
    if not os.path.exists(HASAR_MODEL_PATH):
        eslesen = [p for p in hasar_adaylari if en_iyi_id in p]
        HASAR_MODEL_PATH = eslesen[0] if eslesen else hasar_adaylari[0]
else:
    # JSON yok — ilk modeli al
    HASAR_MODEL_PATH = hasar_adaylari[0]
    en_iyi_id = HASAR_MODEL_PATH.split('/')[-3].replace('hasar_', '')

# EN_IYI sözlüğü (Hücre 9'daki TRAMER raporu bunu kullanıyor)
if en_iyi_id in sonuclar:
    EN_IYI = sonuclar[en_iyi_id]
    EN_IYI['path'] = HASAR_MODEL_PATH
else:
    EN_IYI = {
        'ad':            en_iyi_id,
        'mask_mAP50':    0.0,
        'inference_ms':  0.0,
        'model_mb':      os.path.getsize(HASAR_MODEL_PATH) / 1e6,
        'path':          HASAR_MODEL_PATH,
    }
    print('\n⚠️  Karşılaştırma JSON\'u yok — metrik bilgileri boş kalacak.')
    print('   TRAMER raporunda mAP değeri 0 görünür (sorun değil, sadece bilgi).')

print(f'\n✅ HASAR MODELİ: {EN_IYI["ad"]}')
print(f'   Path: {HASAR_MODEL_PATH}')

# ══ 4. TEST VERİSİ YOLU (Hücre 8/10 için gerekli) ══════════════════════════════
print('\n' + '─' * 70)
if 'DATASET_DIR' not in globals() or not os.path.exists(globals().get('DATASET_DIR', '')):
    # Colab'da indirilmiş dataset ara
    olasi = glob.glob('/content/*/test/images') + glob.glob('/content/*/*/test/images')
    if olasi:
        DATASET_DIR = os.path.dirname(os.path.dirname(olasi[0]))
        print(f'📂 Dataset bulundu: {DATASET_DIR}')
    else:
        DATASET_DIR = None
        print('⚠️  Dataset bulunamadı (session yeni açılmış olabilir).')
        print('   Hücre 8/10\'da kendi görsellerini kullanacaksan sorun değil:')
        print('   TEST_GORSEL = "/content/foto.jpg"  şeklinde yol ver.')
        print('   Test setini kullanacaksan Hücre 4\'ü çalıştırıp veriyi indir.')
else:
    print(f'📂 Dataset hazır: {DATASET_DIR}')

# ══ 5. MODELLERİ GERÇEKTEN YÜKLE VE TEST ET ════════════════════════════════════
print('\n' + '─' * 70)
print('🔄 Modeller belleğe yükleniyor...')

import numpy as np
dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

if KONUM_MODEL_PATH:
    _km = YOLO(KONUM_MODEL_PATH)
    _km.predict(dummy, verbose=False)
    print(f'   ✅ Konum modeli çalışıyor ({len(_km.names)} sınıf)')

_hm = YOLO(HASAR_MODEL_PATH)
_hm.predict(dummy, verbose=False)
print(f'   ✅ Hasar modeli çalışıyor ({_hm.names})')

print('\n' + '=' * 70)
print('🎉 HAZIR — Şimdi HÜCRE 7\'yi çalıştır (füzyon motoru)')
print('=' * 70)
print('   Hücre 7  → Füzyon fonksiyonları')
print('   Hücre 8  → Tek görsel testi')
print('   Hücre 9  → TRAMER PDF raporu')
print('   Hücre 10 → Toplu işlem')
print('=' * 70)

---
## 🔗 HÜCRE 7: Modelleri Yükle + Füzyon Fonksiyonları

Bu hücre iki modeli birleştirip **"Tampon-Kirik"** formatında çıktı üreten mantığı kurar.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔗 HÜCRE 7: FÜZYON MOTORU
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 FÜZYON MANTIĞI:
#    1. Model 1 → araç parçalarını maskeler (Tampon, Kaput, Kapi...)
#    2. Model 2 → hasarları maskeler (Cizik, Gocuk, Kirik)
#    3. Her hasar maskesi, hangi parça maskesiyle en çok örtüşüyor?
#       → "Tampon-Kirik", "Camurluk-Cizik"
#
# 📐 ÖRTÜŞME METRİĞİ:
#    Overlap = (hasar ∩ parça) / hasar_alanı
#    Yani hasarın yüzde kaçı o parçanın üstünde?
#    IoU yerine bu kullanılır çünkü hasar parçadan çok küçüktür.
# ═══════════════════════════════════════════════════════════════════════════════

from ultralytics import YOLO
import numpy as np, cv2

# ─── MODELLERİ YÜKLE ──────────────────────────────────────────────────────────
print('📦 Modeller yükleniyor...')
konum_model = YOLO(KONUM_MODEL_PATH)
hasar_model = YOLO(HASAR_MODEL_PATH)
print(f'   ✅ Model 1 (Konum): {len(konum_model.names)} sınıf')
print(f'   ✅ Model 2 (Hasar): {hasar_model.names}')


def _maske_al(results, H, W):
    """YOLO sonucundan (sınıf, güven, maske, bbox) listesi çıkarır."""
    cikti = []
    if results.masks is None:
        return cikti
    for box, mask in zip(results.boxes, results.masks):
        m = mask.data[0].cpu().numpy()
        if m.shape != (H, W):
            m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
        cikti.append({
            'cls':  results.names[int(box.cls[0])],
            'conf': float(box.conf[0]),
            'mask': m.astype(bool),
            'bbox': box.xyxy[0].cpu().numpy().astype(int).tolist(),
        })
    return cikti


def parca_tespit(img_np, conf=KONUM_CONF):
    """Model 1: Araç parçalarını tespit et, Türkçe isimlere çevir."""
    H, W = img_np.shape[:2]
    res = konum_model.predict(img_np, conf=conf, iou=0.45,
                              verbose=False, retina_masks=True)[0]
    parcalar = []
    for d in _maske_al(res, H, W):
        key = d['cls'].lower().replace(' ', '_').replace('-', '_')
        tr = KONUM_MAP.get(key)
        if tr is None:
            # Kısmi eşleşme dene
            for k, v in KONUM_MAP.items():
                if k in key or key in k:
                    tr = v
                    break
        if tr:
            d['parca'] = tr
            parcalar.append(d)
    return parcalar


def hasar_tespit(img_np, conf=HASAR_CONF):
    """Model 2: Hasarları tespit et."""
    H, W = img_np.shape[:2]
    res = hasar_model.predict(img_np, conf=conf, iou=0.45,
                              verbose=False, retina_masks=True)[0]
    hasarlar = []
    for d in _maske_al(res, H, W):
        d['hasar'] = d['cls']
        hasarlar.append(d)
    return hasarlar


def _ortusme(hasar_mask, parca_mask):
    """Hasarın yüzde kaçı bu parçanın üstünde?"""
    kesisim = np.logical_and(hasar_mask, parca_mask).sum()
    hasar_alan = hasar_mask.sum()
    return kesisim / hasar_alan if hasar_alan > 0 else 0.0


def fuzyon_analiz(img_np, verbose=False):
    """ANA FONKSİYON: İki modeli birleştirip 'Tampon-Kirik' formatı üretir.

    Döner: {
        'parcalar': [...],
        'hasarlar': [...],
        'birlesik': [{'etiket': 'Tampon-Kirik', 'parca': 'Tampon',
                      'hasar': 'Kirik', 'guven': 0.87, 'alan_px': 4210,
                      'alan_yuzde': 2.4, 'ortusme': 0.91, 'mask': ...}],
        'goruntu_boyut': (W, H),
    }
    """
    H, W = img_np.shape[:2]
    toplam_px = H * W

    parcalar = parca_tespit(img_np)
    hasarlar = hasar_tespit(img_np)

    if verbose:
        print(f'   Parça: {[p["parca"] for p in parcalar]}')
        print(f'   Hasar: {[h["hasar"] for h in hasarlar]}')

    birlesik = []
    for h in hasarlar:
        en_iyi_parca, en_iyi_ort = None, 0.0
        for p in parcalar:
            o = _ortusme(h['mask'], p['mask'])
            if o > en_iyi_ort:
                en_iyi_ort, en_iyi_parca = o, p

        if en_iyi_parca and en_iyi_ort >= FUZYON_IOU:
            parca_adi = en_iyi_parca['parca']
            guven = h['conf'] * en_iyi_parca['conf']   # birleşik güven
        else:
            parca_adi = 'Belirsiz'
            guven = h['conf'] * 0.5                     # parça bulunamadı, cezalandır

        alan_px = int(h['mask'].sum())
        birlesik.append({
            'etiket':     f'{parca_adi}-{h["hasar"]}',
            'parca':      parca_adi,
            'hasar':      h['hasar'],
            'guven':      float(guven),
            'hasar_guven':float(h['conf']),
            'alan_px':    alan_px,
            'alan_yuzde': alan_px / toplam_px * 100,
            'ortusme':    float(en_iyi_ort),
            'bbox':       h['bbox'],
            'mask':       h['mask'],
        })

    # Güvene göre sırala
    birlesik.sort(key=lambda x: -x['guven'])

    if verbose:
        print(f'   Birleşik: {[b["etiket"] for b in birlesik]}')

    return {
        'parcalar': parcalar,
        'hasarlar': hasarlar,
        'birlesik': birlesik,
        'goruntu_boyut': (W, H),
    }


print('\n✅ Füzyon motoru hazır:')
print('   - parca_tespit(img)     → Model 1')
print('   - hasar_tespit(img)     → Model 2')
print('   - fuzyon_analiz(img)    → ANA FONKSİYON (Tampon-Kirik üretir)')

---
## 🖼️ HÜCRE 8: Görsel Çıktı — Tek Fotoğraf Testi

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🖼️ HÜCRE 8: GÖRSELLEŞTİRME
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 3 PANELLİ ÇIKTI:
#    Panel 1 → Model 1: Tespit edilen parçalar
#    Panel 2 → Model 2: Tespit edilen hasarlar
#    Panel 3 → FÜZYON: Tampon-Kirik etiketleriyle birleşik sonuç
#
# 🎛️ KULLANIM:
#    TEST_GORSEL = '/content/foto.jpg'  (kendi dosyan)
#    veya URL ver, veya boş bırak → test setinden rastgele seçer
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np, requests, io, random, os
from pathlib import Path

TEST_GORSEL = None   # None = test setinden rastgele seç

# ─── GÖRSELİ YÜKLE ────────────────────────────────────────────────────────────
if TEST_GORSEL is None:
    test_dir = os.path.join(DATASET_DIR, 'test', 'images')
    adaylar = list(Path(test_dir).glob('*.*'))
    TEST_GORSEL = str(random.choice(adaylar))
    print(f'🎲 Rastgele seçildi: {os.path.basename(TEST_GORSEL)}')

if TEST_GORSEL.startswith('http'):
    r = requests.get(TEST_GORSEL, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
    img_np = np.array(Image.open(io.BytesIO(r.content)).convert('RGB'))
else:
    img_np = np.array(Image.open(TEST_GORSEL).convert('RGB'))

print(f'📐 Boyut: {img_np.shape[1]}x{img_np.shape[0]}')

# ─── ANALİZ ───────────────────────────────────────────────────────────────────
print('\n🔍 Analiz ediliyor...')
sonuc = fuzyon_analiz(img_np, verbose=True)

# ─── RENK PALETİ ──────────────────────────────────────────────────────────────
PARCA_RENK = {
    'Tampon':     [0.20, 0.60, 1.00],
    'Kaput':      [1.00, 0.55, 0.15],
    'Kapi':       [0.35, 0.85, 0.45],
    'Camurluk':   [0.85, 0.35, 0.85],
    'Bagaj':      [1.00, 0.85, 0.25],
    'Far':        [0.25, 0.90, 0.90],
    'Cam':        [0.65, 0.65, 0.95],
    'Dikiz-Ayna': [0.95, 0.45, 0.55],
}
HASAR_RENK = {
    'Cizik': [1.00, 0.25, 0.25],
    'Gocuk': [0.25, 0.55, 1.00],
    'Kirik': [1.00, 0.80, 0.10],
}

def overlay_ciz(ax, img, ogeler, renk_map, anahtar, alpha=0.45):
    ax.imshow(img)
    H, W = img.shape[:2]
    ov = np.zeros((H, W, 4), dtype=np.float32)
    for o in ogeler:
        c = renk_map.get(o[anahtar], [0.5, 0.5, 0.5])
        ov[o['mask'], :3] = c
        ov[o['mask'],  3] = alpha
    ax.imshow(ov)
    ax.axis('off')

# ─── 3 PANELLİ GÖRSEL ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# Panel 1: Parçalar
overlay_ciz(axes[0], img_np, sonuc['parcalar'], PARCA_RENK, 'parca')
axes[0].set_title(f'MODEL 1 — Parça Tespiti ({len(sonuc["parcalar"])} adet)',
                  fontsize=13, fontweight='bold')
if sonuc['parcalar']:
    seen = {p['parca'] for p in sonuc['parcalar']}
    axes[0].legend(handles=[mpatches.Patch(color=PARCA_RENK.get(s, [.5]*3), label=s)
                            for s in sorted(seen)],
                   loc='upper right', fontsize=9, framealpha=0.9)

# Panel 2: Hasarlar
overlay_ciz(axes[1], img_np, sonuc['hasarlar'], HASAR_RENK, 'hasar', alpha=0.55)
axes[1].set_title(f'MODEL 2 — Hasar Tespiti ({len(sonuc["hasarlar"])} adet)',
                  fontsize=13, fontweight='bold')
if sonuc['hasarlar']:
    seen = {h['hasar'] for h in sonuc['hasarlar']}
    axes[1].legend(handles=[mpatches.Patch(color=HASAR_RENK.get(s, [.5]*3), label=s)
                            for s in sorted(seen)],
                   loc='upper right', fontsize=9, framealpha=0.9)

# Panel 3: Füzyon
axes[2].imshow(img_np)
H, W = img_np.shape[:2]
ov = np.zeros((H, W, 4), dtype=np.float32)
for b in sonuc['birlesik']:
    c = HASAR_RENK.get(b['hasar'], [0.5, 0.5, 0.5])
    ov[b['mask'], :3] = c
    ov[b['mask'],  3] = 0.6
axes[2].imshow(ov)

# Etiketleri yaz
for b in sonuc['birlesik']:
    x1, y1, x2, y2 = b['bbox']
    axes[2].text(x1, max(y1 - 8, 12), f"{b['etiket']}  %{b['guven']*100:.0f}",
                 fontsize=10, fontweight='bold', color='white',
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='#111827',
                           alpha=0.85, edgecolor='none'))
axes[2].set_title(f'FÜZYON — Birleşik Sonuç ({len(sonuc["birlesik"])} hasar)',
                  fontsize=13, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
kayit = f'{GORSEL_DIR}/analiz_{os.path.basename(TEST_GORSEL).split(".")[0]}.png'
plt.savefig(kayit, dpi=180, bbox_inches='tight')
plt.show()

# ─── METİN ÖZET ───────────────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('📋 TESPİT SONUÇLARI')
print('=' * 70)
if sonuc['birlesik']:
    for i, b in enumerate(sonuc['birlesik'], 1):
        print(f'{i}. {b["etiket"]:25s} güven: %{b["guven"]*100:5.1f} | '
              f'alan: %{b["alan_yuzde"]:.2f} | örtüşme: %{b["ortusme"]*100:.0f}')
else:
    print('   Hasar tespit edilmedi.')
print('=' * 70)
print(f'💾 {kayit}')

---
## 📄 HÜCRE 9: TRAMER Raporu — PDF Çıktısı

Sigorta ekspertiz belgesi formatında, Türkçe karakter destekli PDF rapor üretir.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📄 HÜCRE 9: TRAMER RAPORU ÜRETİCİ
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 RAPOR İÇERİĞİ:
#    - Başlık + belge no + tarih
#    - Araç bilgileri (plaka, marka, model — manuel girilir)
#    - Tespit edilen hasar tablosu (Parça | Hasar Tipi | Alan | Güven)
#    - Analiz görseli (gömülü)
#    - Sistem bilgisi + yasal not
#
# 🔤 TÜRKÇE KARAKTER:
#    DejaVuSans fontu indirilir (ReportLab varsayılan fontlarında ğ/ş/ı yok)
# ═══════════════════════════════════════════════════════════════════════════════

import os, urllib.request, io
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                TableStyle, Image as RLImage)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from PIL import Image as PILImage

# ─── TÜRKÇE FONT KUR ──────────────────────────────────────────────────────────
FONT_DIR = '/content/fonts'
os.makedirs(FONT_DIR, exist_ok=True)
FONT_REG  = f'{FONT_DIR}/DejaVuSans.ttf'
FONT_BOLD = f'{FONT_DIR}/DejaVuSans-Bold.ttf'

if not os.path.exists(FONT_REG):
    print('📥 Türkçe font indiriliyor...')
    base = 'https://raw.githubusercontent.com/dejavu-fonts/dejavu-fonts/master/build/ttf'
    try:
        urllib.request.urlretrieve(f'{base}/DejaVuSans.ttf', FONT_REG)
        urllib.request.urlretrieve(f'{base}/DejaVuSans-Bold.ttf', FONT_BOLD)
    except Exception:
        # Yedek: sistem fontu
        !apt-get install -y fonts-dejavu-core -qq
        import shutil as _sh
        _sh.copy('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', FONT_REG)
        _sh.copy('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', FONT_BOLD)

pdfmetrics.registerFont(TTFont('DejaVu',      FONT_REG))
pdfmetrics.registerFont(TTFont('DejaVu-Bold', FONT_BOLD))
print('✅ Türkçe font hazır')


def tramer_raporu_olustur(sonuc, img_np, arac_bilgi=None, dosya_adi=None):
    """Füzyon sonucundan TRAMER formatında PDF rapor üretir.

    arac_bilgi: {'plaka':..., 'marka':..., 'model':..., 'yil':...,
                 'sase':..., 'police_no':..., 'hasar_no':...}
    """
    if arac_bilgi is None:
        arac_bilgi = {}

    simdi = datetime.now()
    belge_no = arac_bilgi.get('hasar_no') or f"HT-{simdi.strftime('%Y%m%d-%H%M%S')}"

    if dosya_adi is None:
        dosya_adi = f'{RAPOR_DIR}/tramer_{belge_no}.pdf'

    # ── STİLLER ──────────────────────────────────────────────────────────────
    ss = getSampleStyleSheet()
    st_baslik = ParagraphStyle('B', parent=ss['Title'], fontName='DejaVu-Bold',
                                fontSize=17, textColor=colors.HexColor('#0f172a'),
                                spaceAfter=2, alignment=1)
    st_alt    = ParagraphStyle('A', parent=ss['Normal'], fontName='DejaVu',
                                fontSize=9.5, textColor=colors.HexColor('#64748b'),
                                alignment=1, spaceAfter=12)
    st_h2     = ParagraphStyle('H', parent=ss['Heading2'], fontName='DejaVu-Bold',
                                fontSize=11.5, textColor=colors.HexColor('#1e40af'),
                                spaceBefore=13, spaceAfter=6)
    st_p      = ParagraphStyle('P', parent=ss['Normal'], fontName='DejaVu',
                                fontSize=9, leading=13)
    st_kucuk  = ParagraphStyle('K', parent=ss['Normal'], fontName='DejaVu',
                                fontSize=7.5, textColor=colors.HexColor('#64748b'),
                                leading=11)

    doc = SimpleDocTemplate(dosya_adi, pagesize=A4,
                            leftMargin=18*mm, rightMargin=18*mm,
                            topMargin=15*mm, bottomMargin=15*mm)
    story = []

    # ── BAŞLIK ───────────────────────────────────────────────────────────────
    story.append(Paragraph('ARAÇ HASAR TESPİT RAPORU', st_baslik))
    story.append(Paragraph('Yapay Zekâ Destekli Otomatik Ekspertiz Sistemi', st_alt))

    # ── BELGE BİLGİLERİ ──────────────────────────────────────────────────────
    belge_tablo = Table([
        ['Belge No', belge_no, 'Düzenleme Tarihi', simdi.strftime('%d.%m.%Y %H:%M')],
        ['Poliçe No', arac_bilgi.get('police_no', '—'),
         'Rapor Türü', 'Otomatik Ön Tespit'],
    ], colWidths=[32*mm, 46*mm, 38*mm, 46*mm])
    belge_tablo.setStyle(TableStyle([
        ('FONTNAME', (0,0), (-1,-1), 'DejaVu'),
        ('FONTNAME', (0,0), (0,-1), 'DejaVu-Bold'),
        ('FONTNAME', (2,0), (2,-1), 'DejaVu-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 8.5),
        ('BACKGROUND', (0,0), (0,-1), colors.HexColor('#f1f5f9')),
        ('BACKGROUND', (2,0), (2,-1), colors.HexColor('#f1f5f9')),
        ('GRID', (0,0), (-1,-1), 0.4, colors.HexColor('#cbd5e1')),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ]))
    story.append(belge_tablo)

    # ── ARAÇ BİLGİLERİ ───────────────────────────────────────────────────────
    story.append(Paragraph('1. ARAÇ BİLGİLERİ', st_h2))
    arac_tablo = Table([
        ['Plaka', arac_bilgi.get('plaka', '—'), 'Marka', arac_bilgi.get('marka', '—')],
        ['Model', arac_bilgi.get('model', '—'), 'Model Yılı', str(arac_bilgi.get('yil', '—'))],
        ['Şasi No', arac_bilgi.get('sase', '—'), 'Renk', arac_bilgi.get('renk', '—')],
    ], colWidths=[32*mm, 46*mm, 38*mm, 46*mm])
    arac_tablo.setStyle(TableStyle([
        ('FONTNAME', (0,0), (-1,-1), 'DejaVu'),
        ('FONTNAME', (0,0), (0,-1), 'DejaVu-Bold'),
        ('FONTNAME', (2,0), (2,-1), 'DejaVu-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 8.5),
        ('BACKGROUND', (0,0), (0,-1), colors.HexColor('#f8fafc')),
        ('BACKGROUND', (2,0), (2,-1), colors.HexColor('#f8fafc')),
        ('GRID', (0,0), (-1,-1), 0.4, colors.HexColor('#cbd5e1')),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ]))
    story.append(arac_tablo)

    # ── HASAR TABLOSU ────────────────────────────────────────────────────────
    story.append(Paragraph('2. TESPİT EDİLEN HASARLAR', st_h2))

    birlesik = sonuc['birlesik']
    if birlesik:
        veri = [['#', 'Parça', 'Hasar Tipi', 'Alan (%)', 'Güven (%)', 'Durum']]
        for i, b in enumerate(birlesik, 1):
            if b['guven'] >= 0.70:
                durum = 'Kesin'
            elif b['guven'] >= 0.45:
                durum = 'Olası'
            else:
                durum = 'Şüpheli'
            veri.append([
                str(i), b['parca'], b['hasar'],
                f"{b['alan_yuzde']:.2f}",
                f"{b['guven']*100:.1f}",
                durum,
            ])

        hasar_tablo = Table(veri, colWidths=[10*mm, 38*mm, 32*mm, 26*mm, 26*mm, 30*mm])
        stil = [
            ('FONTNAME', (0,0), (-1,0), 'DejaVu-Bold'),
            ('FONTNAME', (0,1), (-1,-1), 'DejaVu'),
            ('FONTSIZE', (0,0), (-1,-1), 8.5),
            ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1e40af')),
            ('TEXTCOLOR', (0,0), (-1,0), colors.white),
            ('ALIGN', (0,0), (0,-1), 'CENTER'),
            ('ALIGN', (3,0), (-1,-1), 'CENTER'),
            ('GRID', (0,0), (-1,-1), 0.4, colors.HexColor('#cbd5e1')),
            ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
            ('TOPPADDING', (0,0), (-1,-1), 5),
            ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ]
        for i in range(1, len(veri)):
            if i % 2 == 0:
                stil.append(('BACKGROUND', (0,i), (-1,i), colors.HexColor('#f8fafc')))
        hasar_tablo.setStyle(TableStyle(stil))
        story.append(hasar_tablo)

        # Özet
        toplam_alan = sum(b['alan_yuzde'] for b in birlesik)
        parcalar_str = ', '.join(sorted({b['parca'] for b in birlesik}))
        tipler_str   = ', '.join(sorted({b['hasar'] for b in birlesik}))
        story.append(Spacer(1, 7))
        story.append(Paragraph(
            f'<b>Özet:</b> Toplam <b>{len(birlesik)}</b> hasar tespit edildi. '
            f'Etkilenen parçalar: <b>{parcalar_str}</b>. '
            f'Hasar tipleri: <b>{tipler_str}</b>. '
            f'Toplam hasarlı alan: <b>%{toplam_alan:.2f}</b>.', st_p))
    else:
        story.append(Paragraph('Görsel üzerinde herhangi bir hasar tespit edilmemiştir.', st_p))

    # ── GÖRSEL ───────────────────────────────────────────────────────────────
    story.append(Paragraph('3. GÖRSEL ANALİZ', st_h2))

    # Overlay'li görsel hazırla
    vis = img_np.astype(np.float32).copy()
    for b in birlesik:
        c = np.array(HASAR_RENK.get(b['hasar'], [0.5, 0.5, 0.5])) * 255
        vis[b['mask']] = vis[b['mask']] * 0.42 + c * 0.58
    vis = vis.astype(np.uint8)

    tmp_img = '/content/_rapor_gorsel.jpg'
    PILImage.fromarray(vis).save(tmp_img, 'JPEG', quality=88)

    ih, iw = vis.shape[:2]
    max_w = 160 * mm
    max_h = 105 * mm
    oran = min(max_w / iw, max_h / ih)
    story.append(RLImage(tmp_img, width=iw * oran, height=ih * oran))
    story.append(Spacer(1, 4))

    lejant = ' | '.join(f'{k}' for k in HASAR_RENK)
    story.append(Paragraph(f'Renk kodları: {lejant}', st_kucuk))

    # ── SİSTEM BİLGİSİ ───────────────────────────────────────────────────────
    story.append(Paragraph('4. SİSTEM BİLGİSİ', st_h2))
    sistem_tablo = Table([
        ['Parça Tespit Modeli', 'YOLOv11m-seg (Carparts-Seg)'],
        ['Hasar Tespit Modeli', EN_IYI['ad']],
        ['Hasar Modeli mAP50', f"{EN_IYI['mask_mAP50']:.4f}"],
        ['İşlem Süresi', f"~{EN_IYI['inference_ms']:.0f} ms/görsel"],
        ['Görüntü Çözünürlüğü', f"{sonuc['goruntu_boyut'][0]} x {sonuc['goruntu_boyut'][1]} px"],
    ], colWidths=[58*mm, 104*mm])
    sistem_tablo.setStyle(TableStyle([
        ('FONTNAME', (0,0), (-1,-1), 'DejaVu'),
        ('FONTNAME', (0,0), (0,-1), 'DejaVu-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 8),
        ('BACKGROUND', (0,0), (0,-1), colors.HexColor('#f8fafc')),
        ('GRID', (0,0), (-1,-1), 0.4, colors.HexColor('#cbd5e1')),
        ('TOPPADDING', (0,0), (-1,-1), 4),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
    ]))
    story.append(sistem_tablo)

    # ── YASAL NOT ────────────────────────────────────────────────────────────
    story.append(Spacer(1, 10))
    story.append(Paragraph(
        '<b>ÖNEMLİ AÇIKLAMA:</b> Bu rapor yapay zekâ tabanlı görüntü analiz sistemi '
        'tarafından otomatik olarak üretilmiştir ve <b>ön değerlendirme</b> niteliğindedir. '
        'Nihai hasar tespiti, onarım kapsamı ve tazminat tutarı yetkili eksper '
        'incelemesi sonucunda belirlenir. Sistem çıktıları yalnızca eksper iş akışını '
        'hızlandırmak amacıyla referans olarak kullanılmalıdır. '
        'Görüntü kalitesi, çekim açısı ve ışık koşulları tespit doğruluğunu etkileyebilir.',
        st_kucuk))

    doc.build(story)
    return dosya_adi


# ─── TEST: Hücre 8'deki görsel için rapor üret ────────────────────────────────
print('\n📄 Örnek rapor üretiliyor...')
pdf = tramer_raporu_olustur(
    sonuc, img_np,
    arac_bilgi={
        'plaka':     '34 ABC 123',
        'marka':     'Volkswagen',
        'model':     'Passat',
        'yil':       2020,
        'renk':      'Beyaz',
        'sase':      'WVWZZZ3CZLE000000',
        'police_no': 'POL-2026-004521',
    }
)
print(f'✅ Rapor hazır: {pdf}')

# Colab'da indir
try:
    from google.colab import files
    files.download(pdf)
except Exception:
    pass

---
## 📦 HÜCRE 10: Toplu İşlem — Klasördeki Tüm Görseller

Birden fazla fotoğrafı işleyip her biri için görsel + PDF rapor üretir, sonuçları Excel'e yazar.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📦 HÜCRE 10: TOPLU İŞLEM
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NE YAPAR?
#    - Belirtilen klasördeki tüm görselleri işler
#    - Her biri için: analiz görseli + TRAMER PDF üretir
#    - Tüm sonuçları tek Excel dosyasında toplar
#    - Hepsini ZIP'ler ve indirir
#
# 🎛️ AYARLAR:
#    KAYNAK_KLASOR : İşlenecek görsellerin olduğu klasör
#    LIMIT         : Kaç görsel işlenecek (None = hepsi)
#    PDF_URET      : Her görsel için PDF üretilsin mi
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd, shutil, time
from pathlib import Path
from PIL import Image
import numpy as np

KAYNAK_KLASOR = os.path.join(DATASET_DIR, 'test', 'images')   # veya '/content/fotograflar'
LIMIT         = 20        # None = hepsi
PDF_URET      = True
GORSEL_URET   = True

gorseller = sorted([p for p in Path(KAYNAK_KLASOR).glob('*.*')
                    if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
if LIMIT:
    gorseller = gorseller[:LIMIT]

print('=' * 70)
print(f'📦 TOPLU İŞLEM — {len(gorseller)} görsel')
print('=' * 70)

kayitlar = []
t0 = time.time()

for i, gp in enumerate(gorseller, 1):
    try:
        img = np.array(Image.open(gp).convert('RGB'))
        s = fuzyon_analiz(img)

        # Excel için kayıt
        if s['birlesik']:
            for b in s['birlesik']:
                kayitlar.append({
                    'Dosya':      gp.name,
                    'Etiket':     b['etiket'],
                    'Parça':      b['parca'],
                    'Hasar Tipi': b['hasar'],
                    'Güven (%)':  round(b['guven'] * 100, 1),
                    'Alan (%)':   round(b['alan_yuzde'], 3),
                    'Örtüşme(%)': round(b['ortusme'] * 100, 1),
                })
        else:
            kayitlar.append({
                'Dosya': gp.name, 'Etiket': '—', 'Parça': '—',
                'Hasar Tipi': 'Tespit yok', 'Güven (%)': 0,
                'Alan (%)': 0, 'Örtüşme(%)': 0,
            })

        # Analiz görseli
        if GORSEL_URET and s['birlesik']:
            vis = img.astype(np.float32).copy()
            for b in s['birlesik']:
                c = np.array(HASAR_RENK.get(b['hasar'], [0.5]*3)) * 255
                vis[b['mask']] = vis[b['mask']] * 0.42 + c * 0.58
            Image.fromarray(vis.astype(np.uint8)).save(
                f'{GORSEL_DIR}/{gp.stem}_analiz.jpg', 'JPEG', quality=88)

        # PDF rapor
        if PDF_URET:
            tramer_raporu_olustur(
                s, img,
                arac_bilgi={'hasar_no': f'HT-{gp.stem[:20]}'},
                dosya_adi=f'{RAPOR_DIR}/rapor_{gp.stem}.pdf'
            )

        if i % 5 == 0 or i == len(gorseller):
            gecen = time.time() - t0
            hiz = i / gecen
            kalan = (len(gorseller) - i) / hiz if hiz > 0 else 0
            print(f'   {i}/{len(gorseller)} işlendi | {hiz:.2f} görsel/sn | ~{kalan:.0f}sn kaldı')

    except Exception as e:
        print(f'   ⚠️ {gp.name}: {str(e)[:80]}')

# ─── EXCEL ÖZET ───────────────────────────────────────────────────────────────
df_sonuc = pd.DataFrame(kayitlar)
excel_yol = f'{OUTPUT_DIR}/toplu_analiz_sonuclari.xlsx'

with pd.ExcelWriter(excel_yol, engine='openpyxl') as w:
    df_sonuc.to_excel(w, sheet_name='Tüm Tespitler', index=False)

    # Özet sayfa: parça × hasar çapraz tablo
    gecerli = df_sonuc[df_sonuc['Hasar Tipi'] != 'Tespit yok']
    if not gecerli.empty:
        pivot = pd.crosstab(gecerli['Parça'], gecerli['Hasar Tipi'], margins=True,
                            margins_name='TOPLAM')
        pivot.to_excel(w, sheet_name='Parça-Hasar Matrisi')

        ozet = gecerli.groupby('Etiket').agg(
            Adet=('Etiket', 'count'),
            Ort_Guven=('Güven (%)', 'mean'),
            Ort_Alan=('Alan (%)', 'mean'),
        ).round(2).sort_values('Adet', ascending=False)
        ozet.to_excel(w, sheet_name='Etiket Özeti')

print('\n' + '=' * 70)
print('✅ TOPLU İŞLEM TAMAMLANDI')
print('=' * 70)
print(f'   İşlenen görsel : {len(gorseller)}')
print(f'   Toplam tespit  : {len(df_sonuc[df_sonuc["Hasar Tipi"] != "Tespit yok"])}')
print(f'   Süre           : {(time.time()-t0)/60:.1f} dk')
print(f'\n   📊 Excel   : {excel_yol}')
print(f'   📄 Raporlar: {RAPOR_DIR}')
print(f'   🖼️  Görseller: {GORSEL_DIR}')

# En sık tespitler
gecerli = df_sonuc[df_sonuc['Hasar Tipi'] != 'Tespit yok']
if not gecerli.empty:
    print(f'\n🏷️  EN SIK TESPİT EDİLEN ETİKETLER:')
    for etiket, adet in gecerli['Etiket'].value_counts().head(10).items():
        print(f'   {etiket:28s}: {adet}')

# ─── ZIP + İNDİR ──────────────────────────────────────────────────────────────
print('\n📦 Çıktılar arşivleniyor...')
zip_yol = shutil.make_archive('/content/tez_ciktilari', 'zip', OUTPUT_DIR)
print(f'✅ {zip_yol}')

try:
    from google.colab import files
    files.download(zip_yol)
except Exception:
    print('   (Colab dışı ortam — manuel indir)')

---
## 🎯 HÜCRE 6C: Uçtan Uca Füzyon Başarısı

İki model ayrı ayrı iyi olabilir ama **birleştirince** ne kadar başarılı? Bu, tezinin özgün katkısını ölçen bölüm.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 HÜCRE 6C: FÜZYON SİSTEMİ DEĞERLENDİRMESİ
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NE ÖLÇER?
#    Hasar modeli tek başına 'Cizik' diyor. Füzyon 'Tampon-Cizik' diyor.
#    Peki parça ataması ne kadar doğru? Bu hücre onu ölçer.
#
# 📊 ÜRETİLEN METRİKLER:
#    - Parça atama oranı (kaç hasara parça atanabildi)
#    - Ortalama örtüşme skoru
#    - Etiket dağılımı (Tampon-Cizik: 45 adet, ...)
#    - Parça × Hasar çapraz matrisi
#
# ⚠️ NOT: Bu hücre Hücre 7'den SONRA çalıştırılmalı (füzyon fonksiyonları lazım)
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import Counter
import time

DEGERLENDIRME_SAYISI = 150   # Test setinden kaç görsel değerlendirilecek

test_dir = os.path.join(DATASET_DIR, 'test', 'images')
test_gorseller = sorted(Path(test_dir).glob('*.*'))[:DEGERLENDIRME_SAYISI]

print('=' * 70)
print(f'🎯 FÜZYON DEĞERLENDİRMESİ — {len(test_gorseller)} test görseli')
print('=' * 70)

etiket_sayaci   = Counter()
parca_sayaci    = Counter()
hasar_sayaci    = Counter()
ortusme_skorlar = []
atanan, atanamayan, hasarsiz = 0, 0, 0
sureler = []

for i, gp in enumerate(test_gorseller, 1):
    try:
        img = np.array(Image.open(gp).convert('RGB'))
        t0 = time.time()
        s = fuzyon_analiz(img)
        sureler.append((time.time() - t0) * 1000)

        if not s['birlesik']:
            hasarsiz += 1
            continue

        for b in s['birlesik']:
            etiket_sayaci[b['etiket']] += 1
            parca_sayaci[b['parca']]   += 1
            hasar_sayaci[b['hasar']]   += 1
            ortusme_skorlar.append(b['ortusme'])
            if b['parca'] == 'Belirsiz':
                atanamayan += 1
            else:
                atanan += 1

        if i % 25 == 0:
            print(f'   {i}/{len(test_gorseller)} işlendi...')
    except Exception as e:
        print(f'   ⚠️ {gp.name}: {str(e)[:60]}')

toplam_hasar = atanan + atanamayan

# ══ RAPOR ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('📊 FÜZYON PERFORMANSI')
print('=' * 70)
print(f'   Değerlendirilen görsel   : {len(test_gorseller)}')
print(f'   Hasar bulunmayan görsel  : {hasarsiz}')
print(f'   Toplam tespit edilen hasar: {toplam_hasar}')
print()
if toplam_hasar > 0:
    print(f'   ✅ Parça ATANDI          : {atanan:4d}  (%{atanan/toplam_hasar*100:.1f})')
    print(f'   ⚠️  Parça ATANAMADI      : {atanamayan:4d}  (%{atanamayan/toplam_hasar*100:.1f})')
if ortusme_skorlar:
    print()
    print(f'   Ortalama örtüşme skoru   : %{np.mean(ortusme_skorlar)*100:.1f}')
    print(f'   Medyan örtüşme skoru     : %{np.median(ortusme_skorlar)*100:.1f}')
if sureler:
    print(f'   Ortalama işlem süresi    : {np.mean(sureler):.0f} ms/görsel')
    print(f'   (2 model + füzyon dahil)')

print(f'\n🏷️  EN SIK BİRLEŞİK ETİKETLER:')
for et, ad in etiket_sayaci.most_common(15):
    bar = '█' * int(ad / max(etiket_sayaci.values()) * 30)
    print(f'   {et:26s}: {ad:4d} {bar}')

# ══ GRAFİKLER ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Füzyon Sistemi Analizi', fontsize=15, fontweight='bold')

# 1. Etiket dağılımı
ax = axes[0, 0]
top = etiket_sayaci.most_common(12)
if top:
    ax.barh([t[0] for t in top][::-1], [t[1] for t in top][::-1], color='#3b82f6')
    ax.set_xlabel('Adet'); ax.set_title('En Sık Birleşik Etiketler')
    ax.grid(axis='x', alpha=0.3)

# 2. Hasar tipi dağılımı
ax = axes[0, 1]
if hasar_sayaci:
    renkler = ['#ef4444', '#3b82f6', '#f59e0b']
    ax.pie(hasar_sayaci.values(), labels=hasar_sayaci.keys(), autopct='%1.1f%%',
           colors=renkler[:len(hasar_sayaci)], startangle=90,
           wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax.set_title('Hasar Tipi Dağılımı')

# 3. Parça dağılımı
ax = axes[1, 0]
if parca_sayaci:
    pl = parca_sayaci.most_common()
    ax.bar([p[0] for p in pl], [p[1] for p in pl], color='#10b981')
    ax.set_ylabel('Adet'); ax.set_title('Etkilenen Parça Dağılımı')
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.3)

# 4. Örtüşme skoru histogramı
ax = axes[1, 1]
if ortusme_skorlar:
    ax.hist(np.array(ortusme_skorlar) * 100, bins=25, color='#8b5cf6',
            edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(ortusme_skorlar) * 100, color='red', linestyle='--',
               linewidth=2, label=f'Ortalama: %{np.mean(ortusme_skorlar)*100:.0f}')
    ax.set_xlabel('Örtüşme Skoru (%)'); ax.set_ylabel('Frekans')
    ax.set_title('Hasar-Parça Örtüşme Dağılımı'); ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fuzyon_analizi.png', dpi=180, bbox_inches='tight')
plt.show()

# ══ ÇAPRAZ MATRİS (Parça × Hasar) ══════════════════════════════════════════════
if etiket_sayaci:
    matris = pd.DataFrame(0, index=sorted(parca_sayaci.keys()),
                          columns=HASAR_SINIFLAR)
    for et, ad in etiket_sayaci.items():
        if '-' in et:
            p, h = et.rsplit('-', 1)
            if p in matris.index and h in matris.columns:
                matris.loc[p, h] = ad

    print('\n' + '=' * 70)
    print('📋 PARÇA × HASAR ÇAPRAZ MATRİSİ')
    print('=' * 70)
    print(matris.to_string())

    # Heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matris.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(matris.columns)))
    ax.set_xticklabels(matris.columns)
    ax.set_yticks(range(len(matris.index)))
    ax.set_yticklabels(matris.index)
    for i in range(len(matris.index)):
        for j in range(len(matris.columns)):
            v = matris.values[i, j]
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > matris.values.max()/2 else 'black',
                    fontweight='bold')
    ax.set_title('Parça × Hasar Tipi Dağılımı', fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Adet')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/parca_hasar_matrisi.png', dpi=180, bbox_inches='tight')
    plt.show()

    matris.to_excel(f'{OUTPUT_DIR}/parca_hasar_matrisi.xlsx')

# ══ ÖZET KAYDET ════════════════════════════════════════════════════════════════
ozet = {
    'degerlendirilen_gorsel': len(test_gorseller),
    'toplam_hasar_tespiti':   toplam_hasar,
    'parca_atanan':           atanan,
    'parca_atanamayan':       atanamayan,
    'atama_orani_yuzde':      round(atanan/toplam_hasar*100, 2) if toplam_hasar else 0,
    'ort_ortusme_yuzde':      round(float(np.mean(ortusme_skorlar))*100, 2) if ortusme_skorlar else 0,
    'ort_islem_suresi_ms':    round(float(np.mean(sureler)), 1) if sureler else 0,
    'benzersiz_etiket_sayisi': len(etiket_sayaci),
}
with open(f'{OUTPUT_DIR}/fuzyon_ozet.json', 'w') as f:
    json.dump(ozet, f, indent=2, ensure_ascii=False)

print(f'\n💾 Kaydedildi:')
print(f'   {OUTPUT_DIR}/fuzyon_analizi.png')
print(f'   {OUTPUT_DIR}/parca_hasar_matrisi.png')
print(f'   {OUTPUT_DIR}/parca_hasar_matrisi.xlsx')
print(f'   {OUTPUT_DIR}/fuzyon_ozet.json')
print('=' * 70)